# Step 5 — L4 VisDrone2019-MOT — No ReID

Same Step-5 structure; only the dataset branch is changed to VisDrone2019-MOT test-dev. No auxiliary ReID neural model is used.

In [ ]:
# 1. Mount Google Drive
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if not IN_COLAB:
    raise RuntimeError("Run this controlled notebook in Google Colab.")

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
MOUNTED_DRIVE_ROOT = Path("/content/drive/MyDrive")
if not MOUNTED_DRIVE_ROOT.exists():
    raise FileNotFoundError(MOUNTED_DRIVE_ROOT)
print("Mounted output Drive:", MOUNTED_DRIVE_ROOT)


Mounted at /content/drive
Mounted output Drive: /content/drive/MyDrive


## 2. Install controlled dependencies
PyTorch/CUDA from Colab are preserved. The remaining dependencies are installed before
loading tracker/model code.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "ultralytics==8.4.116",
    "lap==0.5.13",
    "gdown==6.1.0",
    "onnx>=1.17.0",
    "PyYAML>=6.0",
    "pandas>=2.0",
    "numpy>=1.26",
    "scipy>=1.11",
    "tqdm>=4.66",
    "pillow>=10.0",
    "requests>=2.31",
    "matplotlib>=3.8",
    "onnxruntime-gpu==1.24.4",
    "faster-coco-eval>=1.6.7",
    "nvidia-ml-py>=12.560.30",
    "tabulate>=0.9",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed", *PACKAGES
])


import importlib

try:
    import tensorrt as trt
except Exception:
    import torch as _torch_for_trt_install

    cuda_text = str(_torch_for_trt_install.version.cuda or "")
    package = "tensorrt-cu12" if cuda_text.startswith("12") else "tensorrt"

    print(f"TensorRT Python package not found. Installing {package}...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        package,
    ])

    importlib.invalidate_caches()
    import tensorrt as trt

import cv2
import gdown
import onnx
import numpy as np
import pandas as pd
import requests
import scipy
import torch
import torchvision
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an NVIDIA GPU runtime in Colab.")

import logging
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*['\"]half['\"] is deprecated.*",
)

try:
    from ultralytics.utils import LOGGER as ULTRALYTICS_LOGGER

    class _SuppressUltralyticsHalfDeprecation(logging.Filter):
        def filter(self, record):
            try:
                message = record.getMessage()
            except Exception:
                return True

            return not (
                "'half' is deprecated" in message
                or '"half" is deprecated' in message
            )

    ULTRALYTICS_LOGGER.addFilter(
        _SuppressUltralyticsHalfDeprecation()
    )

except Exception:
    pass

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA        :", torch.version.cuda)
print("Ultralytics :", ultralytics.__version__)
print("gdown       :", getattr(gdown, "__version__", "unknown"))
print("ONNX        :", onnx.__version__)
print("TensorRT    :", trt.__version__)
print("OpenCV      :", cv2.__version__)
print("GPU         :", torch.cuda.get_device_name(0))


TensorRT Python package not found. Installing tensorrt-cu12...
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python      : 3.12.13
PyTorch     : 2.11.0+cu128
Torchvision : 0.26.0+cu128
CUDA        : 12.8
Ultralytics : 8.4.116
gdown       : 6.1.0
ONNX        : 1.22.0
TensorRT    : 11.2.1.2
OpenCV      : 5.0.0
GPU         : NVIDIA L4


## 3. USER CONFIGURATION — edit only this cell
Paste the three final model paths or shared-file URLs below.

Supported source examples:
- `/content/drive/MyDrive/.../best.pt`
- `https://drive.google.com/file/d/.../view?usp=sharing`
- a generic direct `https://.../model.pt` URL

`TEST_4K` is the recommended final Okutama run (official test-video archive, about 4 GB).
`SAMPLE_4K` is a faster one-video smoke/full-pipeline run (about 540 MB).
`CUSTOM` accepts a local folder/ZIP or a downloadable ZIP in OKUTAMA_CUSTOM_SOURCE.

In [ ]:
# --------------------------- MODEL SOURCES ---------------------------
YOLO26S_MODEL_SOURCE = "https://drive.google.com/file/d/10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r/view?usp=drive_link"
RTDETR_MODEL_SOURCE = "https://drive.google.com/file/d/10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u/view?usp=drive_link"
BPD_MODEL_SOURCE = "https://drive.google.com/file/d/1cbf-pONCQaaEtzLndOxPxomltRDjmv2a/view?usp=drive_link"
# Model format handling.
# "auto" is strongly recommended for .pt/.pth/.onnx links.
# Use "engine" explicitly only when the source really is a TensorRT engine.
MODEL_FORMAT_HINTS = {
    "YOLO26s": "auto",
    "RT-DETR-R18": "auto",
    "BPD-YOLOn/L-FPN": "auto",
}

# Optional filename substring filters.
# Leave empty for normal direct file links.
# If you paste a Google Drive FOLDER link containing several model files,
# set a unique substring here so the intended file can be selected.
MODEL_FILENAME_CONTAINS = {
    "YOLO26s": "",
    "RT-DETR-R18": "",
    "BPD-YOLOn/L-FPN": "",
}

# Never trust an old v1 cache merely because its file size is non-zero.
# v2 stores files in a source-hash cache and validates them before reuse.
VALIDATE_MODEL_BINARY_BEFORE_RUN = True


# --------------------------- VIDEO SELECTION ---------------------------
# Change only this value for another reproducible 5-video set.
VIDEO_SELECTION_SEED = 100

# --------------------------- OUTPUT ---------------------------
OUTPUT_RELATIVE_DIR = "aerial_human_detection/step5_tracking_final"
RUN_TAG = f"visdrone_mot_l4_tuned_s{VIDEO_SELECTION_SEED}"

# --------------------------- HARDWARE / DETECTOR ---------------------------
REQUIRE_L4 = True
DEVICE_ID = 0
IMAGE_SIZE = 1280
MAX_DETECTIONS = 3000
DETECTOR_CONF_THRESHOLDS = {
    "YOLO26s": 0.20,
    "RT-DETR-R18": 0.45,
    "BPD-YOLOn/L-FPN": 0.20,
}
DETECTOR_NMS_IOU = 0.70
DETECTION_EVAL_IOU = 0.50
USE_FP16_CHECKPOINT_INFERENCE = False

# The deprecated Ultralytics half predict option is intentionally not passed.
# TensorRT engines keep their exported precision automatically.
WARMUP_RUNS = 3

# --------------------------- TRACKERS ---------------------------
#
# ID-STABLE PROFILE
#
# IMPORTANT:
# TRACK_LOST_GRACE_SECONDS does NOT forcibly lock an identity.
# It keeps a lost track alive long enough to be re-associated before it
# is removed. This is safer than forcing an old ID onto a new detection.
#
# Ultralytics 8.4.116 uses track_buffer directly as a frame count, so the
# effective buffer is calculated dynamically as:
#
#       round(video_fps * TRACK_LOST_GRACE_SECONDS)
#
# Example: 30 FPS × 2 seconds = 60 frames.
#

TRACKERS_TO_RUN = [
    "ByteTrack",
    "BoT-SORT",
]

TRACK_LOST_GRACE_SECONDS = 2.0

# Shared association policy.
# A slightly stricter matching threshold is used for crowded VisDrone scenes
# to reduce incorrect cross-identity associations. Conservative new-track
# thresholds reduce premature creation of replacement IDs.
TRACKER_COMMON_SETTINGS = {
    "match_thresh": 0.80,
    "fuse_score": True,
}

# Model-aware confidence gates.
#
# These are tracker association thresholds, not detector thresholds.
# Detector thresholds remain:
#   YOLO26s = 0.20
#   RT-DETR-R18 = 0.45
#   BPD-YOLOn/L-FPN = 0.20
#
# Low-stage thresholds are aligned with the fixed detector operating
# thresholds. Detector thresholds themselves are unchanged.
TRACKER_MODEL_THRESHOLDS = {
    "YOLO26s": {
        "track_high_thresh": 0.30,
        "track_low_thresh": 0.20,
        "new_track_thresh": 0.35,
    },
    "RT-DETR-R18": {
        "track_high_thresh": 0.52,
        "track_low_thresh": 0.45,
        "new_track_thresh": 0.56,
    },
    "BPD-YOLOn/L-FPN": {
        "track_high_thresh": 0.30,
        "track_low_thresh": 0.20,
        "new_track_thresh": 0.35,
    },
}

# ByteTrack keeps motion/IoU association only.
BYTETRACK_SETTINGS = {
    **TRACKER_COMMON_SETTINGS,
}

# BoT-SORT:
# - GMC remains enabled for moving aerial cameras.
# - ReID is enabled to help distinguish identities after occlusion.
# - proximity_thresh is relaxed moderately so appearance can help after
#   displacement, while appearance_thresh is strict to avoid identity swaps.
BOTSORT_SETTINGS = {
    **TRACKER_COMMON_SETTINGS,
    "gmc_method": "sparseOptFlow",
    "proximity_thresh": 0.35,
    "appearance_thresh": 0.85,
    "with_reid": False,
}

# Tracking quality is the Step-5 objective.
# Runtime remains internal only; Step 4 is the deployment benchmark.
REPORT_RUNTIME_METRICS_IN_STEP5 = False

# --------------------------- REID POLICY ---------------------------
# No auxiliary ReID / appearance neural model is used.
ENABLE_GLOBAL_ID_RECOVERY = False

# --------------------------- VISDRONE2019-MOT ---------------------------
DATASET_PROFILE = "TEST_DEV"

# Official VisDrone2019-MOT test-dev Google Drive archive.
VISDRONE_MOT_SOURCE = (
    "https://drive.google.com/file/d/"
    "14z8Acxopj1d86-qhsF1NwS4Bv3KYa4Wu/view?usp=sharing"
)
VISDRONE_MOT_SPLIT = "test_dev"

# Person-only target: official VisDrone category 1 = pedestrian.
VISDRONE_PERSON_CATEGORY_IDS = {1}

# Used for preview MP4 playback and 2-second track-buffer conversion.
VISDRONE_EVAL_FPS = 30.0

MAX_VIDEOS = 5
PREFER_DISTINCT_SCENARIOS = True
SEGMENTS_PER_VIDEO = 1
SEGMENT_LENGTH_FRAMES = 300
SEGMENT_STRIDE_FRAMES = 300
MIN_MEAN_PERSONS_PER_FRAME = 1.0

# --------------------------- EXPORT / RESUME ---------------------------
SAVE_ANNOTATED_VIDEO = True
SHOW_GROUND_TRUTH_ON_VIDEO = True
SHOW_TRACK_TRAILS = True
TRACK_TRAIL_LENGTH = 30
SAVE_DETECTION_CACHE_JSONL = True
SAVE_FRAME_CSV = True
SAVE_TRACK_JSONL = True
RESUME_COMPLETED_DETECTION_CACHE = True
RESUME_COMPLETED_TRACKER_RUNS = True
CONTINUE_ON_SYSTEM_ERROR = False
FAIL_IF_FINAL_MATRIX_INCOMPLETE = True

# --------------------------- PROPOSAL REFERENCE CHECKS ---------------------------
# Confirm these against the signed proposal before final report sign-off.
# They are reporting gates, never tuning targets on Okutama.
TARGET_MOTA = 0.50
TARGET_IDF1 = 0.60
TARGET_IDSW_PER_100_FRAMES = 5.0

# --------------------------- PINNED/PRIMARY SOURCES ---------------------------
RTDETR_REPO_URL = "https://github.com/lyuwenyu/RT-DETR.git"
RTDETR_REPO_COMMIT = "199fc382f53abbfb5c1804c97b0e8b204e3cb8d0"
TRACKEVAL_REPO_URL = "https://github.com/JonathonLuiten/TrackEval.git"

OKUTAMA_OFFICIAL_URLS = {
    "SAMPLE_4K": (
        "https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/"
        "AKx_1WK7YqOIf4I0PDX5vRk/Sample.zip"
        "?dl=1&e=1&rlkey=7u7131amaul29amyr4jbnnu03"
    ),
    "TEST_4K": (
        "https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/"
        "AKym5JciNZCmrCHfQkfoBtU/TestSetVideos.zip"
        "?dl=1&e=1&rlkey=7u7131amaul29amyr4jbnnu03"
    ),
}

print("=" * 96)
print("STEP-5 CONFIGURATION")
print("=" * 96)
print("Dataset profile     :", f"VisDrone2019-MOT {VISDRONE_MOT_SPLIT}")
print("Maximum videos      :", MAX_VIDEOS)
print("Segments per video  :", SEGMENTS_PER_VIDEO)
print("Frames per segment  :", SEGMENT_LENGTH_FRAMES)
print("Image size          :", IMAGE_SIZE)
print("Detector confidence thresholds:")
for _model_name, _threshold in DETECTOR_CONF_THRESHOLDS.items():
    print(f"  {_model_name:20s}: {_threshold:.2f}")
print("Trackers            :", TRACKERS_TO_RUN)
print("Lost-track grace    :", f"{TRACK_LOST_GRACE_SECONDS:.1f} seconds")
print("BoT-SORT ReID       : DISABLED")
print("Video selection seed:", VIDEO_SELECTION_SEED)
print("Videos per run      :", MAX_VIDEOS)
print("External ReID model : NONE")
print("Require NVIDIA L4   :", REQUIRE_L4)
print("=" * 96)

STEP-5 CONFIGURATION
Dataset profile     : VisDrone2019-MOT test_dev
Maximum videos      : 5
Segments per video  : 1
Frames per segment  : 300
Image size          : 1280
Detector confidence thresholds:
  YOLO26s             : 0.20
  RT-DETR-R18         : 0.45
  BPD-YOLOn/L-FPN     : 0.20
Trackers            : ['ByteTrack', 'BoT-SORT']
Lost-track grace    : 2.0 seconds
BoT-SORT ReID       : DISABLED
Video selection seed: 100
Videos per run      : 5
External ReID model : NONE
Require NVIDIA L4   : True


## Video selection

Change only `VIDEO_SELECTION_SEED` to choose another reproducible set of five VisDrone2019-MOT sequences.

## Identity policy — no ReID

The 2-second lost-track buffer is enabled. No neural appearance model is used.

## Final operating thresholds and warning policy

The final L4 run uses fixed detector confidence thresholds:

- **YOLO26s:** `0.20`
- **RT-DETR-R18:** `0.45`
- **BPD-YOLOn/L-FPN:** `0.20`

These thresholds are applied before tracker association and are saved in the run audit.

The deprecated Ultralytics half predict option is not passed anywhere in this version.
TensorRT engines retain the precision with which they were exported. A narrow log filter also
suppresses only that specific legacy deprecation message if an internal Ultralytics layer emits it;
other warnings remain visible.

Because the confidence thresholds changed from the earlier `0.08` run, this version uses a new
`RUN_TAG`, so old detection caches cannot be reused accidentally.


## 4. Runtime audit, persistent output tree, and shared helpers

In [ ]:

import contextlib
import gc
import hashlib
import importlib
import json
import math
import os
import platform
import re
import shlex
import shutil
import time
import traceback
import urllib.parse
import zipfile
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from IPython.display import display

OUTPUT_ROOT = MOUNTED_DRIVE_ROOT / OUTPUT_RELATIVE_DIR / RUN_TAG
AUDIT_DIR = OUTPUT_ROOT / "00_audit"
MODEL_META_DIR = OUTPUT_ROOT / "01_models"
DATA_META_DIR = OUTPUT_ROOT / "02_dataset"
RUNS_DIR = OUTPUT_ROOT / "03_runs"
TRACKEVAL_DIR = OUTPUT_ROOT / "04_trackeval"
METRICS_DIR = OUTPUT_ROOT / "05_metrics"
PLOTS_DIR = OUTPUT_ROOT / "06_plots"
REPORT_ASSETS_DIR = OUTPUT_ROOT / "07_report_assets"

for p in [OUTPUT_ROOT, AUDIT_DIR, MODEL_META_DIR, DATA_META_DIR, RUNS_DIR,
          TRACKEVAL_DIR, METRICS_DIR, PLOTS_DIR, REPORT_ASSETS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_CACHE = Path("/content/step5_model_cache_v5")
DATA_CACHE = Path("/content/step5_visdrone_mot_cache")
TOOL_CACHE = Path("/content/step5_tools")
for p in [MODEL_CACHE, DATA_CACHE, TOOL_CACHE]:
    p.mkdir(parents=True, exist_ok=True)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def safe_slug(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))
    value = re.sub(r"_+", "_", value).strip("_.")
    return value or "item"


def percentile(values: Sequence[float], q: float) -> float:
    arr = np.asarray(list(values), dtype=np.float64)
    return float(np.percentile(arr, q)) if arr.size else float("nan")


def nanmean(values: Sequence[float]) -> float:
    arr = np.asarray(list(values), dtype=np.float64)
    return float(np.nanmean(arr)) if arr.size else float("nan")


def is_http_url(value: str) -> bool:
    return isinstance(value, str) and value.lower().startswith(("http://", "https://"))


def is_google_drive_url(value: str) -> bool:
    return is_http_url(value) and ("drive.google.com" in value.lower() or "docs.google.com" in value.lower())


def is_placeholder(value: str) -> bool:
    text = str(value).strip().upper()
    return (not text) or text.startswith("PASTE_") or "PUT_YOUR" in text


def detector_conf_threshold(model_name: str) -> float:
    """Return the fixed Step-5 confidence threshold for a detector."""
    if model_name not in DETECTOR_CONF_THRESHOLDS:
        raise KeyError(
            f"No detector confidence threshold configured for: {model_name}"
        )

    return float(DETECTOR_CONF_THRESHOLDS[model_name])


def cuda_sync() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize(DEVICE_ID)


def reset_cuda_peak() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE_ID)


def peak_cuda_mib() -> float:
    return float(torch.cuda.max_memory_allocated(DEVICE_ID) / (1024 ** 2)) if torch.cuda.is_available() else 0.0



def normalize_download_url(url: str) -> str:
    """
    Normalize cloud URLs for direct binary download.
    Dropbox links are forced to dl=1.
    """
    url = str(url).strip()

    if not is_http_url(url):
        return url

    parsed = urllib.parse.urlsplit(url)

    if "dropbox.com" in parsed.netloc.lower():
        query = urllib.parse.parse_qs(
            parsed.query,
            keep_blank_values=True,
        )
        query["dl"] = ["1"]

        flat_query = {
            key: values[-1]
            for key, values in query.items()
        }

        url = urllib.parse.urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                parsed.path,
                urllib.parse.urlencode(flat_query),
                parsed.fragment,
            )
        )

    return url


def http_download_robust(
    url: str,
    destination: Path,
    minimum_bytes: int = 1_000_000,
) -> Path:
    """
    Robust large-file downloader for Colab.

    - follows redirects
    - retries transient failures
    - resumes from a .part file when supported
    - retries from scratch if resume is rejected
    - atomically finalizes the download
    - rejects tiny HTML/error responses
    """
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    url = normalize_download_url(url)

    part_path = destination.with_name(
        destination.name + ".part"
    )

    print("Resolved download URL:", url)
    print("Destination          :", destination)

    resume_cmd = [
        "curl",
        "-L",
        "--fail",
        "--show-error",
        "--retry",
        "5",
        "--retry-delay",
        "3",
        "--retry-all-errors",
        "--connect-timeout",
        "30",
        "-C",
        "-",
        "-o",
        str(part_path),
        url,
    ]

    result = subprocess.run(
        resume_cmd,
        check=False,
    )

    if result.returncode != 0:
        print(
            "Resume mode failed or is unsupported. "
            "Retrying the download from scratch..."
        )

        if part_path.exists():
            part_path.unlink()

        fresh_cmd = [
            "curl",
            "-L",
            "--fail",
            "--show-error",
            "--retry",
            "5",
            "--retry-delay",
            "3",
            "--retry-all-errors",
            "--connect-timeout",
            "30",
            "-o",
            str(part_path),
            url,
        ]

        subprocess.check_call(
            fresh_cmd
        )

    if not part_path.exists():
        raise RuntimeError(
            f"Download finished without creating: {part_path}"
        )

    size_bytes = part_path.stat().st_size

    print(
        "Downloaded size      : "
        f"{size_bytes:,} bytes "
        f"({size_bytes / (1024 ** 2):.2f} MiB)"
    )

    if size_bytes < int(minimum_bytes):
        with part_path.open("rb") as f:
            preview_bytes = f.read(2048)

        preview = preview_bytes.decode(
            "utf-8",
            errors="replace",
        )[:500]

        raise RuntimeError(
            "Downloaded response is unexpectedly small and "
            "is probably an HTML/error response rather than "
            "the requested binary file. "
            f"size_bytes={size_bytes:,}; preview={preview!r}"
        )

    if destination.exists():
        destination.unlink()

    part_path.replace(
        destination
    )

    return destination


GPU_NAME = torch.cuda.get_device_name(DEVICE_ID)
if REQUIRE_L4 and "L4" not in GPU_NAME.upper():
    raise RuntimeError(f"Controlled Step-5 requires NVIDIA L4; active GPU: {GPU_NAME}")

environment = {
    "timestamp_utc": utc_now_iso(),
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_total_memory_gib": torch.cuda.get_device_properties(DEVICE_ID).total_memory / (1024 ** 3),
    "ultralytics": ultralytics.__version__,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "rtdetr_repo": RTDETR_REPO_URL,
    "rtdetr_commit": RTDETR_REPO_COMMIT,
    "trackeval_repo": TRACKEVAL_REPO_URL,
    "model_source_hotfix": "v5_tensorrt_plus_visdrone_mot_download",
    "reid_policy": "NO_ADDITIONAL_REID_MODEL",
}
write_json(AUDIT_DIR / "environment.json", environment)

with (AUDIT_DIR / "pip_freeze.txt").open("w", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, stderr=subprocess.STDOUT, text=True, check=False)
with (AUDIT_DIR / "nvidia_smi.txt").open("w", encoding="utf-8") as f:
    subprocess.run(["nvidia-smi"], stdout=f, stderr=subprocess.STDOUT, text=True, check=False)

execution_config = {
    "output_root": str(OUTPUT_ROOT),
    "run_tag": RUN_TAG,
    "models": ["YOLO26s", "RT-DETR-R18", "BPD-YOLOn/L-FPN"],
    "trackers": TRACKERS_TO_RUN,
    "image_size": IMAGE_SIZE,
    "detector_conf_thresholds": DETECTOR_CONF_THRESHOLDS,
    "detector_nms_iou": DETECTOR_NMS_IOU,
    "max_detections": MAX_DETECTIONS,
    "detection_eval_iou": DETECTION_EVAL_IOU,
    "dataset_profile": DATASET_PROFILE,
    "visdrone_mot_split": VISDRONE_MOT_SPLIT,
    "visdrone_person_category_ids": sorted(VISDRONE_PERSON_CATEGORY_IDS),
    "visdrone_eval_fps": VISDRONE_EVAL_FPS,
    "video_selection_seed": VIDEO_SELECTION_SEED,
    "max_videos": MAX_VIDEOS,
    "segments_per_video": SEGMENTS_PER_VIDEO,
    "segment_length_frames": SEGMENT_LENGTH_FRAMES,
    "segment_stride_frames": SEGMENT_STRIDE_FRAMES,
    "bytetrack_base_settings": BYTETRACK_SETTINGS,
    "botsort_base_settings": BOTSORT_SETTINGS,
    "tracker_model_thresholds": TRACKER_MODEL_THRESHOLDS,
    "track_lost_grace_seconds": TRACK_LOST_GRACE_SECONDS,
    "botsort_reid_enabled": False,
    "external_reid_model_used": False,
    "private_final_test_used": False,
}
write_json(AUDIT_DIR / "execution_config.json", execution_config)
print("GPU        :", GPU_NAME)
print("Output root:", OUTPUT_ROOT)


GPU        : NVIDIA L4
Output root: /content/drive/MyDrive/aerial_human_detection/step5_tracking_final/visdrone_mot_l4_tuned_s100


## 5. Resolve and verify the three model artifacts — v4 TensorRT support

The supplied Google Drive links are the TensorRT artifacts produced in Step 4.

v4 supports:
- PyTorch checkpoint (`.pt` / `.pth`)
- ONNX (`.onnx`)
- TensorRT (`.engine`)

It recognizes both TensorRT containers used by this project:
- Ultralytics metadata-wrapped engine for YOLO26s and BPD-YOLOn/L-FPN.
- Raw TensorRT `ftrt` engine for RT-DETR-R18.

For detected TensorRT engines, SHA-256 is compared with the hashes recorded in the Step-4 report.


In [ ]:
MODEL_SOURCES = {
    "YOLO26s": YOLO26S_MODEL_SOURCE,
    "RT-DETR-R18": RTDETR_MODEL_SOURCE,
    "BPD-YOLOn/L-FPN": BPD_MODEL_SOURCE,
}

MODEL_DEFAULT_BASENAMES = {
    "YOLO26s": "yolo26s_step5_model",
    "RT-DETR-R18": "rtdetr_r18_step5_model",
    "BPD-YOLOn/L-FPN": "bpd_yolon_lfpn_step5_model",
}

ALLOWED_MODEL_FORMATS = {
    "YOLO26s": {"pt", "onnx", "engine"},
    "RT-DETR-R18": {"pth", "onnx", "engine"},
    "BPD-YOLOn/L-FPN": {"pt", "onnx", "engine"},
}

KNOWN_STEP4_ENGINE_SHA256 = {
    "YOLO26s": "fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af6347602bb382a63b5dfa",
    "RT-DETR-R18": "9d9a2ad34cad42e3b8c80299887552d29729448dda2b278345c3cc3b92257b51",
    "BPD-YOLOn/L-FPN": "518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e343004e5e9b3e2e06112",
}



def _source_hash(source: str) -> str:
    return hashlib.sha256(str(source).strip().encode("utf-8")).hexdigest()[:16]


def _first_bytes(path: Path, n: int = 64) -> bytes:
    with path.open("rb") as f:
        return f.read(n)


def _is_probably_html_or_text_error(path: Path) -> Tuple[bool, str]:
    head = _first_bytes(path, 1024)
    low = head.lower()

    signatures = [
        b"<!doctype html",
        b"<html",
        b"<head",
        b"<body",
        b"access denied",
        b"permission denied",
        b"google drive",
        b"quota exceeded",
        b"virus scan warning",
    ]

    if any(sig in low for sig in signatures):
        preview = head.decode("utf-8", errors="replace")[:300]
        return True, preview

    return False, ""


def _torch_archive_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    """
    Structural Torch checkpoint probe without unpickling custom project classes.
    Modern torch.save files are ZIP archives containing pickle metadata.
    Legacy pickle-based torch files commonly start with pickle protocol 0x80.
    """
    info = {
        "zipfile": False,
        "zip_integrity": None,
        "pickle_metadata": False,
        "legacy_pickle_header": False,
    }

    head = _first_bytes(path, 16)

    # Legacy pickle-based serialization.
    if len(head) >= 1 and head[0] == 0x80:
        info["legacy_pickle_header"] = True
        return True, info

    if not zipfile.is_zipfile(path):
        return False, info

    info["zipfile"] = True

    try:
        with zipfile.ZipFile(path, "r") as zf:
            bad_member = zf.testzip()
            info["zip_integrity"] = (bad_member is None)
            names = zf.namelist()
            info["pickle_metadata"] = any(
                name.endswith(".pkl")
                or name.endswith("data.pkl")
                or "/data.pkl" in name
                for name in names
            )
            info["zip_member_sample"] = names[:10]

            if bad_member is not None:
                info["bad_zip_member"] = bad_member
                return False, info

            if info["pickle_metadata"]:
                return True, info

    except Exception as exc:
        info["zip_error"] = repr(exc)
        return False, info

    return False, info


def _onnx_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    info = {}

    try:
        model = onnx.load(str(path), load_external_data=False)
        info["ir_version"] = int(model.ir_version)
        info["graph_name"] = str(model.graph.name)
        info["num_nodes"] = int(len(model.graph.node))
        info["num_inputs"] = int(len(model.graph.input))
        info["num_outputs"] = int(len(model.graph.output))

        # check_model may raise on malformed/truncated protobuf.
        onnx.checker.check_model(model)
        info["checker"] = "PASSED"

        del model
        return True, info

    except Exception as exc:
        info["error"] = f"{type(exc).__name__}: {exc}"
        return False, info


def _ultralytics_engine_metadata_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    info = {}

    try:
        file_size = path.stat().st_size

        if file_size < 16:
            return False, {"error": "file too small"}

        with path.open("rb") as f:
            length_bytes = f.read(4)

            if len(length_bytes) != 4:
                return False, {"error": "missing metadata length"}

            meta_len = int.from_bytes(
                length_bytes,
                byteorder="little",
                signed=True,
            )
            info["metadata_length"] = int(meta_len)

            if not (1 <= meta_len <= min(16 * 1024 * 1024, file_size - 8)):
                return False, info

            meta_bytes = f.read(meta_len)

            try:
                metadata = json.loads(meta_bytes.decode("utf-8"))
            except Exception as exc:
                info["metadata_json_error"] = f"{type(exc).__name__}: {exc}"
                return False, info

            if not isinstance(metadata, dict):
                return False, info

            info["description"] = str(metadata.get("description", ""))
            info["task"] = metadata.get("task")
            info["imgsz"] = metadata.get("imgsz")
            info["metadata"] = metadata
            info["engine_payload_offset"] = 4 + meta_len
            info["engine_payload_first_16_hex"] = f.read(16).hex()

            looks_ultralytics = (
                "ultralytics" in info["description"].lower()
                or "task" in metadata
                or "imgsz" in metadata
                or "names" in metadata
            )

            return bool(looks_ultralytics), info

    except Exception as exc:
        info["error"] = f"{type(exc).__name__}: {exc}"
        return False, info


def _raw_tensorrt_magic_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    head = _first_bytes(path, 32)
    info = {
        "first_32_bytes_hex": head.hex(),
        "first_4_ascii": head[:4].decode("latin1", errors="replace"),
    }

    if head[:4] == b"ftrt":
        info["magic"] = "ftrt"
        return True, info

    return False, info


def _tensorrt_engine_probe(path: Path) -> Tuple[bool, Dict[str, Any]]:
    wrapped_ok, wrapped_info = _ultralytics_engine_metadata_probe(path)

    if wrapped_ok:
        return True, {
            "engine_container": "ultralytics_metadata_wrapped",
            "ultralytics_metadata": wrapped_info,
        }

    raw_ok, raw_info = _raw_tensorrt_magic_probe(path)

    if raw_ok:
        return True, {
            "engine_container": "raw_tensorrt",
            "raw_engine": raw_info,
        }

    return False, {
        "ultralytics_metadata_probe": wrapped_info,
        "raw_engine_probe": raw_info,
    }

def detect_model_format(path: Path, requested_hint: str = "auto") -> Tuple[str, Dict[str, Any]]:
    """
    Returns one of: torch, onnx, engine, unknown.
    requested_hint may be auto/pt/pth/onnx/engine.
    """
    requested_hint = str(requested_hint).lower().strip().lstrip(".")
    diagnostics = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "first_32_bytes_hex": _first_bytes(path, 32).hex(),
        "requested_hint": requested_hint,
    }

    is_text_error, preview = _is_probably_html_or_text_error(path)

    if is_text_error:
        diagnostics["text_error_preview"] = preview
        return "unknown", diagnostics

    if requested_hint in {"pt", "pth", "auto"}:
        torch_ok, torch_info = _torch_archive_probe(path)
        diagnostics["torch_probe"] = torch_info

        if torch_ok:
            return "torch", diagnostics

    if requested_hint in {"onnx", "auto"}:
        onnx_ok, onnx_info = _onnx_probe(path)
        diagnostics["onnx_probe"] = onnx_info

        if onnx_ok:
            return "onnx", diagnostics

    if requested_hint in {"engine", "auto"}:
        engine_ok, engine_info = _tensorrt_engine_probe(path)
        diagnostics["engine_probe"] = engine_info

        if engine_ok:
            return "engine", diagnostics

    return "unknown", diagnostics


def canonical_extension(model_name: str, detected_format: str) -> str:
    if detected_format == "onnx":
        return "onnx"

    if detected_format == "engine":
        return "engine"

    if detected_format == "torch":
        return "pth" if model_name == "RT-DETR-R18" else "pt"

    raise ValueError(detected_format)


def _candidate_files(folder: Path) -> List[Path]:
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and p.stat().st_size >= 100_000
        and not p.name.endswith(".part")
    )


def extract_google_drive_file_id(url: str) -> Optional[str]:
    """
    Extract a Google Drive FILE id from common share-link forms.

    Supported examples:
      https://drive.google.com/file/d/FILE_ID/view?usp=drive_link
      https://drive.google.com/open?id=FILE_ID
      https://drive.google.com/uc?id=FILE_ID
    """
    text = str(url).strip()

    patterns = [
        r"/file/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
        r"/d/([A-Za-z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    return None


def extract_google_drive_folder_id(url: str) -> Optional[str]:
    match = re.search(
        r"/folders/([A-Za-z0-9_-]+)",
        str(url).strip(),
    )
    return match.group(1) if match else None


def _download_google_drive(source: str, download_dir: Path) -> List[Path]:
    """
    v3 hotfix:
    Never download the /view page itself.

    For a Drive FILE URL:
      1. extract FILE_ID ourselves;
      2. call gdown.download(id=FILE_ID, output=explicit_binary_path);
      3. reject tiny HTML/permission pages;
      4. let binary preflight determine pt/pth/onnx.

    For a Drive FOLDER URL:
      use gdown.download_folder(id=FOLDER_ID, ...).
    """
    download_dir.mkdir(parents=True, exist_ok=True)

    folder_id = extract_google_drive_folder_id(source)

    if folder_id:
        print("Google Drive folder ID:", folder_id)

        try:
            result = gdown.download_folder(
                id=folder_id,
                output=str(download_dir),
                quiet=False,
            )
        except Exception as exc:
            raise RuntimeError(
                "Google Drive folder download failed. "
                "Make sure the folder is shared with this Colab session. "
                f"Original error: {type(exc).__name__}: {exc}"
            ) from exc

        files = _candidate_files(download_dir)

        if not files:
            raise RuntimeError(
                "Google Drive folder was resolved, but no model-sized file "
                "was downloaded. Check folder permissions and contents."
            )

        return files

    file_id = extract_google_drive_file_id(source)

    if not file_id:
        raise RuntimeError(
            "Could not extract a Google Drive FILE ID from this URL: "
            f"{source}"
        )

    print("Google Drive file ID:", file_id)

    # Explicit neutral extension. We detect the real binary format afterwards.
    target = download_dir / "google_drive_model_download.bin"

    # Remove any stale/interrupted artifact before a fresh download.
    for stale in [
        target,
        target.with_suffix(target.suffix + ".part"),
    ]:
        if stale.exists():
            stale.unlink()

    try:
        result = gdown.download(
            id=file_id,
            output=str(target),
            quiet=False,
        )
    except Exception as exc:
        raise RuntimeError(
            "Google Drive file download failed. "
            "The file must be accessible to this Colab session; for a public "
            "share link, set sharing to 'Anyone with the link'. "
            f"File ID: {file_id}. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc

    if result is None:
        raise RuntimeError(
            "gdown returned no downloaded file. "
            f"Google Drive file ID: {file_id}"
        )

    if not target.exists():
        result_path = Path(str(result))
        if result_path.exists():
            target = result_path

    if not target.exists():
        raise RuntimeError(
            "gdown reported success but the expected downloaded file "
            f"does not exist. File ID: {file_id}"
        )

    size_bytes = target.stat().st_size
    print(f"Downloaded bytes: {size_bytes:,}")

    # These project checkpoints are many MiB. A tiny response is almost
    # certainly an HTML/access/error response rather than the model.
    if size_bytes < 100_000:
        head = _first_bytes(target, 2048)
        preview = head.decode("utf-8", errors="replace")[:500]

        raise RuntimeError(
            "Google Drive returned a tiny file instead of the model "
            f"({size_bytes:,} bytes). "
            "This is usually a permission/login/share-page response. "
            "Set the file sharing permission to 'Anyone with the link' "
            "or use an existing path in the currently mounted Drive. "
            f"File ID: {file_id}. "
            f"Response preview: {preview!r}"
        )

    return [target]

def _download_generic_url(source: str, download_dir: Path) -> List[Path]:
    download_dir.mkdir(parents=True, exist_ok=True)

    # gdown also supports ordinary HTTP/HTTPS URLs and handles redirects.
    old_cwd = Path.cwd()

    try:
        os.chdir(download_dir)
        result = gdown.download(
            url=source,
            output=None,
            quiet=False,
        )
    finally:
        os.chdir(old_cwd)

    if result is not None:
        result_path = Path(result)

        if not result_path.is_absolute():
            result_path = download_dir / result_path.name

        if result_path.exists():
            return [result_path]

    # Fallback to the project's robust HTTP downloader.
    parsed_name = Path(urllib.parse.urlsplit(source).path).name
    fallback_name = parsed_name if parsed_name else "downloaded_model.bin"
    fallback = download_dir / fallback_name

    http_download_robust(source, fallback)
    return [fallback]


def _select_and_validate_candidate(
    model_name: str,
    source: str,
    candidates: List[Path],
) -> Tuple[Path, str, Dict[str, Any]]:
    hint = str(MODEL_FORMAT_HINTS[model_name]).lower().strip().lstrip(".")
    filename_filter = str(MODEL_FILENAME_CONTAINS[model_name]).strip().lower()

    if hint not in {"auto", "pt", "pth", "onnx", "engine"}:
        raise ValueError(
            f"{model_name}: MODEL_FORMAT_HINTS must be auto/pt/pth/onnx/engine; got {hint}"
        )

    candidates = [
        p for p in candidates
        if p.exists() and p.is_file() and p.stat().st_size >= 100_000
    ]

    if filename_filter:
        candidates = [
            p for p in candidates
            if filename_filter in p.name.lower()
        ]

    if not candidates:
        raise RuntimeError(
            f"{model_name}: no model-sized file was found after resolving source: {source}"
        )

    valid = []
    invalid = []

    for candidate in candidates:
        detected, diag = detect_model_format(candidate, hint)

        if detected == "torch":
            allowed = (
                "pth" in ALLOWED_MODEL_FORMATS[model_name]
                or "pt" in ALLOWED_MODEL_FORMATS[model_name]
            )
        else:
            allowed = detected in ALLOWED_MODEL_FORMATS[model_name]

        if detected != "unknown" and allowed:
            valid.append((candidate, detected, diag))
        else:
            invalid.append({
                "candidate": str(candidate),
                "detected_format": detected,
                "diagnostics": diag,
            })

    if len(valid) == 0:
        diagnostic_path = MODEL_META_DIR / f"{safe_slug(model_name)}_invalid_source_diagnostics.json"
        write_json(
            diagnostic_path,
            {
                "model": model_name,
                "source": source,
                "candidates": invalid,
            },
        )

        first = invalid[0] if invalid else {}
        raise RuntimeError(
            f"{model_name}: downloaded/resolved bytes are not a valid supported model file. "
            f"Diagnostics were saved to {diagnostic_path}. "
            f"First candidate: {first.get('candidate')}; "
            f"first bytes: {first.get('diagnostics', {}).get('first_32_bytes_hex')}. "
            "Typical causes: wrong Google Drive link, private/unshared file, a folder link "
            "containing multiple files, or a model format that does not match the configured hint."
        )

    if len(valid) > 1:
        names = [str(item[0]) for item in valid]
        raise RuntimeError(
            f"{model_name}: more than one valid model candidate was found: {names}. "
            "Set MODEL_FILENAME_CONTAINS for this model to a unique part of the intended filename."
        )

    return valid[0]


def resolve_model_source(model_name: str, source: str) -> Tuple[Path, Dict[str, Any]]:
    source = str(source).strip()

    if is_placeholder(source):
        raise ValueError(
            f"{model_name}: source is still a placeholder. "
            "Edit USER CONFIGURATION and paste the final model path or URL."
        )

    source_cache = MODEL_CACHE / safe_slug(model_name) / _source_hash(source)
    source_cache.mkdir(parents=True, exist_ok=True)

    local_candidate = Path(os.path.expanduser(source))

    if local_candidate.exists():
        if local_candidate.is_dir():
            candidates = _candidate_files(local_candidate)
        else:
            candidates = [local_candidate]

        source_mode = "local_path"

    elif is_google_drive_url(source):
        # Reuse v2 cache only after binary validation.
        cached = _candidate_files(source_cache)

        if cached:
            try:
                selected, detected, diag = _select_and_validate_candidate(
                    model_name,
                    source,
                    cached,
                )
                print(f"{model_name}: validated v2 cache: {selected}")
                candidates = [selected]
                source_mode = "validated_v2_cache"
            except Exception:
                shutil.rmtree(source_cache, ignore_errors=True)
                source_cache.mkdir(parents=True, exist_ok=True)
                candidates = _download_google_drive(source, source_cache)
                source_mode = "google_drive_download"
        else:
            candidates = _download_google_drive(source, source_cache)
            source_mode = "google_drive_download"

    elif is_http_url(source):
        cached = _candidate_files(source_cache)

        if cached:
            try:
                selected, detected, diag = _select_and_validate_candidate(
                    model_name,
                    source,
                    cached,
                )
                print(f"{model_name}: validated v2 cache: {selected}")
                candidates = [selected]
                source_mode = "validated_v2_cache"
            except Exception:
                shutil.rmtree(source_cache, ignore_errors=True)
                source_cache.mkdir(parents=True, exist_ok=True)
                candidates = _download_generic_url(source, source_cache)
                source_mode = "http_download"
        else:
            candidates = _download_generic_url(source, source_cache)
            source_mode = "http_download"

    else:
        raise FileNotFoundError(
            f"{model_name}: source is neither an existing local path nor an HTTP/HTTPS URL: {source}"
        )

    selected, detected, diagnostics = _select_and_validate_candidate(
        model_name,
        source,
        candidates,
    )

    ext = canonical_extension(model_name, detected)
    canonical = source_cache / f"{MODEL_DEFAULT_BASENAMES[model_name]}.{ext}"

    # For local paths, always create an isolated temporary cache copy.
    if selected.resolve() != canonical.resolve():
        shutil.copy2(selected, canonical)

    # Validate the canonical copy again after copying.
    detected2, diagnostics2 = detect_model_format(
        canonical,
        "engine" if ext == "engine" else "auto",
    )

    expected_detected = "torch" if ext in {"pt", "pth"} else ext

    if detected2 != expected_detected:
        raise RuntimeError(
            f"{model_name}: canonical cache copy failed binary verification. "
            f"Expected {expected_detected}, detected {detected2}."
        )

    actual_sha256 = sha256_file(canonical)
    reported_engine_sha256 = KNOWN_STEP4_ENGINE_SHA256.get(model_name)

    metadata = {
        "model": model_name,
        "source": source,
        "source_mode": source_mode,
        "selected_original_name": selected.name,
        "resolved_path": str(canonical),
        "resolved_extension": ext,
        "detected_binary_format": detected2,
        "size_bytes": canonical.stat().st_size,
        "size_mib": canonical.stat().st_size / (1024 ** 2),
        "sha256": actual_sha256,
        "reported_step4_engine_sha256": (
            reported_engine_sha256 if detected2 == "engine" else None
        ),
        "matches_reported_step4_engine_sha256": (
            actual_sha256 == reported_engine_sha256
            if detected2 == "engine" and reported_engine_sha256
            else None
        ),
        "diagnostics": diagnostics2,
    }

    return canonical, metadata


# ---------------------------------------------------------------------
# Resolve ALL three sources first and fail before dataset/tracking if
# even one model source is invalid.
# ---------------------------------------------------------------------

MODEL_PATHS = {}
model_manifest_rows = []
MODEL_SOURCE_ERRORS = {}

for model_name, source in MODEL_SOURCES.items():
    print("\n" + "=" * 96)
    print("MODEL SOURCE PREFLIGHT:", model_name)
    if is_google_drive_url(source):
        print("Drive file ID        :", extract_google_drive_file_id(source))
        print("Drive folder ID      :", extract_google_drive_folder_id(source))
    print("=" * 96)

    try:
        resolved, meta = resolve_model_source(model_name, source)
        MODEL_PATHS[model_name] = resolved
        model_manifest_rows.append(meta)

        print("Resolved path :", resolved)
        print("Detected type :", meta["detected_binary_format"])
        print("Extension     :", meta["resolved_extension"])
        print("Size MiB      :", f"{meta['size_mib']:.2f}")
        print("SHA-256       :", meta["sha256"])

    except Exception as exc:
        MODEL_SOURCE_ERRORS[model_name] = {
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(),
        }
        print("FAILED:", model_name)
        print(exc)


write_json(
    MODEL_META_DIR / "model_source_preflight_errors.json",
    MODEL_SOURCE_ERRORS,
)

if model_manifest_rows:
    MODEL_MANIFEST = pd.DataFrame(model_manifest_rows)
    MODEL_MANIFEST.to_csv(
        MODEL_META_DIR / "model_source_manifest.csv",
        index=False,
    )
    write_json(
        MODEL_META_DIR / "model_source_manifest.json",
        model_manifest_rows,
    )

    display(
        MODEL_MANIFEST[
            [
                "model",
                "selected_original_name",
                "resolved_extension",
                "detected_binary_format",
                "size_mib",
                "sha256",
            ]
        ]
    )

if MODEL_SOURCE_ERRORS:
    raise RuntimeError(
        "MODEL SOURCE PREFLIGHT FAILED. "
        "No detector, tracker, or TrackEval stage will be started. "
        f"Failed models: {list(MODEL_SOURCE_ERRORS)}. "
        f"Inspect {MODEL_META_DIR / 'model_source_preflight_errors.json'} "
        "and the per-model invalid_source_diagnostics JSON files."
    )

print("\n" + "=" * 96)
print("MODEL SOURCE PREFLIGHT: PASSED FOR ALL THREE MODELS")
print("=" * 96)


MODEL SOURCE PREFLIGHT: YOLO26s
Drive file ID        : 10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r
Drive folder ID      : None
Google Drive file ID: 10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r


Downloading...
From (original): https://drive.google.com/uc?id=10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r
From (redirected): https://drive.google.com/uc?id=10ZVNqYS2RFHA9EMOSwpu3878Klvtoc8r&confirm=t&uuid=466610f0-5f6e-4404-acf2-a03b334bbc8f
To: /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/google_drive_model_download.bin
100%|██████████| 269M/269M [00:01<00:00, 242MB/s]


Downloaded bytes: 268,992,357
Resolved path : /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/yolo26s_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 256.53
SHA-256       : fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af6347602bb382a63b5dfa

MODEL SOURCE PREFLIGHT: RT-DETR-R18
Drive file ID        : 10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u
Drive folder ID      : None
Google Drive file ID: 10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u


Downloading...
From: https://drive.google.com/uc?id=10bPk4Ht6FSFeUHVIwZ9QI_Ktz6SEM30u
To: /content/step5_model_cache_v5/RT-DETR-R18/6ccc85e16300d7b4/google_drive_model_download.bin
100%|██████████| 66.9M/66.9M [00:00<00:00, 143MB/s]


Downloaded bytes: 66,905,348
Resolved path : /content/step5_model_cache_v5/RT-DETR-R18/6ccc85e16300d7b4/rtdetr_r18_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 63.81
SHA-256       : 9d9a2ad34cad42e3b8c80299887552d29729448dda2b278345c3cc3b92257b51

MODEL SOURCE PREFLIGHT: BPD-YOLOn/L-FPN
Drive file ID        : 1cbf-pONCQaaEtzLndOxPxomltRDjmv2a
Drive folder ID      : None
Google Drive file ID: 1cbf-pONCQaaEtzLndOxPxomltRDjmv2a


Downloading...
From (original): https://drive.google.com/uc?id=1cbf-pONCQaaEtzLndOxPxomltRDjmv2a
From (redirected): https://drive.google.com/uc?id=1cbf-pONCQaaEtzLndOxPxomltRDjmv2a&confirm=t&uuid=bfe5979d-07ed-41cb-ab25-b975e3e289fe
To: /content/step5_model_cache_v5/BPD-YOLOn_L-FPN/a3f98ad3094b61a6/google_drive_model_download.bin
100%|██████████| 158M/158M [00:00<00:00, 233MB/s]


Downloaded bytes: 157,862,810
Resolved path : /content/step5_model_cache_v5/BPD-YOLOn_L-FPN/a3f98ad3094b61a6/bpd_yolon_lfpn_step5_model.engine
Detected type : engine
Extension     : engine
Size MiB      : 150.55
SHA-256       : 518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e343004e5e9b3e2e06112


,model,selected_original_name,resolved_extension,detected_binary_format,size_mib,sha256
0,YOLO26s,google_drive_model_download.bin,engine,engine,256.531102,fc46b73eca3520a3dd29b9d187b8fadcdae23ab215af63...
1,RT-DETR-R18,google_drive_model_download.bin,engine,engine,63.805912,9d9a2ad34cad42e3b8c80299887552d29729448dda2b27...
2,BPD-YOLOn/L-FPN,google_drive_model_download.bin,engine,engine,150.549707,518d8aafccc491a17e6c219fa12f112dbedfd60b3b0e34...



MODEL SOURCE PREFLIGHT: PASSED FOR ALL THREE MODELS


## 6. Resolve and extract VisDrone2019-MOT test-dev

Uses the official VisDrone test-dev archive with public ground truth.

In [ ]:
from pathlib import Path
from typing import Tuple, Dict, Any
import shutil
import zipfile
import subprocess
import sys


def install_hf_download_tools():
    try:
        import huggingface_hub
        return
    except Exception:
        pass

    print("Installing Hugging Face download tools...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "huggingface_hub",
            "hf_xet",
        ],
        check=True,
    )


def download_visdrone_from_huggingface(
    archive_path: Path,
) -> Path:
    """
    Automatic fallback mirror for VisDrone2019-MOT-test-dev.zip.
    No manual browser download is required.
    """

    install_hf_download_tools()

    from huggingface_hub import hf_hub_download

    print("=" * 96)
    print("Google Drive quota reached.")
    print("Falling back automatically to Hugging Face mirror...")
    print("=" * 96)

    cached_file = hf_hub_download(
        repo_id="vnthanh/VisDrone2019-MOT",
        repo_type="dataset",
        filename="VisDrone2019-MOT-test-dev.zip",
        resume_download=True,
    )

    cached_file = Path(
        cached_file
    )

    if not cached_file.exists():
        raise RuntimeError(
            "Hugging Face download finished but the archive was not found."
        )

    if (
        archive_path.exists()
        and archive_path.resolve()
        != cached_file.resolve()
    ):
        archive_path.unlink()

    archive_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Copying downloaded archive into Step-5 cache..."
    )

    shutil.copy2(
        cached_file,
        archive_path,
    )

    return archive_path


def resolve_visdrone_mot_source() -> Tuple[Path, Dict[str, Any]]:
    archive_path = (
        DATA_CACHE
        / "VisDrone2019-MOT-test-dev.zip"
    )

    extract_root = (
        DATA_CACHE
        / "VisDrone2019-MOT-test-dev"
    )

    marker = (
        extract_root
        / ".extraction_complete"
    )

    minimum_expected_size = 1_500_000_000

    # ---------------------------------------------------------
    # 1. Use existing valid cached archive when available
    # ---------------------------------------------------------
    if (
        archive_path.exists()
        and archive_path.stat().st_size
        >= minimum_expected_size
        and zipfile.is_zipfile(
            archive_path
        )
    ):
        print(
            "Using cached VisDrone MOT archive:",
            archive_path,
        )

    else:
        # Remove incomplete/corrupt previous attempts.
        if archive_path.exists():
            print(
                "Removing incomplete/corrupt cached archive..."
            )

            archive_path.unlink()

        # -----------------------------------------------------
        # 2. First try official Google Drive
        # -----------------------------------------------------
        google_drive_success = False

        try:
            import gdown

            print(
                "Downloading official VisDrone2019-MOT test-dev..."
            )

            result = gdown.download(
                url=(
                    "https://drive.google.com/uc?id="
                    "14z8Acxopj1d86-qhsF1NwS4Bv3KYa4Wu"
                ),
                output=str(
                    archive_path
                ),
                quiet=False,
            )

            google_drive_success = bool(
                result is not None
                and archive_path.exists()
                and archive_path.stat().st_size
                >= minimum_expected_size
                and zipfile.is_zipfile(
                    archive_path
                )
            )

        except Exception as exc:
            print()
            print(
                "Official Google Drive download failed:"
            )
            print(
                type(exc).__name__,
                ":",
                exc,
            )

            google_drive_success = False

        # -----------------------------------------------------
        # 3. Automatic Hugging Face fallback
        # -----------------------------------------------------
        if not google_drive_success:
            if archive_path.exists():
                archive_path.unlink()

            download_visdrone_from_huggingface(
                archive_path
            )

    # ---------------------------------------------------------
    # 4. Validate archive
    # ---------------------------------------------------------
    if not archive_path.exists():
        raise FileNotFoundError(
            f"VisDrone archive was not created: {archive_path}"
        )

    archive_size = (
        archive_path.stat().st_size
    )

    print()
    print(
        "Archive size:",
        f"{archive_size / (1024 ** 3):.2f} GiB",
    )

    if (
        archive_size
        < minimum_expected_size
    ):
        raise RuntimeError(
            "VisDrone MOT archive is unexpectedly small: "
            f"{archive_size:,} bytes"
        )

    if not zipfile.is_zipfile(
        archive_path
    ):
        raise RuntimeError(
            "Downloaded VisDrone MOT file is not a valid ZIP."
        )

    print(
        "ZIP structure check: PASSED"
    )

    # ---------------------------------------------------------
    # 5. Extract archive
    # ---------------------------------------------------------
    if not marker.exists():
        if extract_root.exists():
            print(
                "Removing incomplete previous extraction..."
            )

            shutil.rmtree(
                extract_root
            )

        extract_root.mkdir(
            parents=True,
            exist_ok=True,
        )

        print()
        print(
            "Testing ZIP integrity..."
        )

        with zipfile.ZipFile(
            archive_path,
            "r",
        ) as zf:
            bad_member = (
                zf.testzip()
            )

            if bad_member is not None:
                raise RuntimeError(
                    "Corrupt ZIP member detected: "
                    f"{bad_member}"
                )

            print(
                "ZIP integrity: PASSED"
            )

            print()
            print(
                "Extracting VisDrone2019-MOT test-dev..."
            )

            zf.extractall(
                extract_root
            )

        marker.write_text(
            utc_now_iso(),
            encoding="utf-8",
        )

        print(
            "Extraction: COMPLETE"
        )

    else:
        print(
            "Using existing extracted VisDrone MOT dataset:",
            extract_root,
        )

    # ---------------------------------------------------------
    # 6. Dataset sanity check
    # ---------------------------------------------------------
    jpg_count = sum(
        1
        for _ in extract_root.rglob(
            "*.jpg"
        )
    )

    txt_count = sum(
        1
        for _ in extract_root.rglob(
            "*.txt"
        )
    )

    print()
    print(
        "Detected JPG frames :",
        f"{jpg_count:,}",
    )

    print(
        "Detected TXT files  :",
        f"{txt_count:,}",
    )

    if jpg_count == 0:
        raise RuntimeError(
            "No VisDrone image frames were found after extraction."
        )

    if txt_count == 0:
        raise RuntimeError(
            "No VisDrone annotation files were found after extraction."
        )

    # ---------------------------------------------------------
    # 7. Audit metadata
    # ---------------------------------------------------------
    meta = {
        "dataset": "VisDrone2019-MOT",
        "split": VISDRONE_MOT_SPLIT,

        "official_source": (
            "https://drive.google.com/file/d/"
            "14z8Acxopj1d86-qhsF1NwS4Bv3KYa4Wu/view"
        ),

        "automatic_fallback": (
            "HuggingFace:"
            "vnthanh/VisDrone2019-MOT/"
            "VisDrone2019-MOT-test-dev.zip"
        ),

        "archive": str(
            archive_path
        ),

        "archive_size_bytes": int(
            archive_size
        ),

        "archive_sha256": sha256_file(
            archive_path
        ),

        "extract_root": str(
            extract_root
        ),

        "jpg_frame_count": int(
            jpg_count
        ),

        "txt_file_count": int(
            txt_count
        ),

        "gt_public": True,

        "person_category_ids": sorted(
            VISDRONE_PERSON_CATEGORY_IDS
        ),
    }

    return (
        extract_root,
        meta,
    )


# =====================================================================
# RUN
# =====================================================================

VISDRONE_MOT_ROOT, visdrone_source_meta = (
    resolve_visdrone_mot_source()
)

write_json(
    DATA_META_DIR
    / "visdrone_mot_source.json",
    visdrone_source_meta,
)

print()
print("=" * 96)
print("VISDRONE2019-MOT READY")
print("=" * 96)

print(
    "Dataset root:",
    VISDRONE_MOT_ROOT,
)

print(
    "Split       :",
    VISDRONE_MOT_SPLIT,
)

print("=" * 96)


Official Google Drive download failed:
FileURLRetrievalError : Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=14z8Acxopj1d86-qhsF1NwS4Bv3KYa4Wu

but Gdown can't. Please check connections and permissions.
Google Drive quota reached.
Falling back automatically to Hugging Face mirror...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(


VisDrone2019-MOT-test-dev.zip: reconstructing file:   0%|          |  0.00B / 2.29GB            

VisDrone2019-MOT-test-dev.zip: downloading bytes:           |  0.00B            

Copying downloaded archive into Step-5 cache...

Archive size: 2.14 GiB
ZIP structure check: PASSED

Testing ZIP integrity...
ZIP integrity: PASSED

Extracting VisDrone2019-MOT test-dev...
Extraction: COMPLETE

Detected JPG frames : 6,635
Detected TXT files  : 17

VISDRONE2019-MOT READY
Dataset root: /content/step5_visdrone_mot_cache/VisDrone2019-MOT-test-dev
Split       : test_dev


## 7. Parse VisDrone MOT GT, pair frame sequences, and select five clips

Official MOT rows use:
`frame_index,target_id,bbox_left,bbox_top,bbox_width,bbox_height,score,object_category,truncation,occlusion`.

For this person-only experiment, only official category `1` (`pedestrian`) with valid GT score is retained.
One continuous 300-frame clip is chosen from each of five seeded source sequences.


In [ ]:
def select_seeded_visdrone_segments(
    all_candidates: pd.DataFrame,
    max_videos: int,
    seed: int,
    prefer_distinct_scenarios: bool = True,
) -> pd.DataFrame:
    if all_candidates.empty:
        raise RuntimeError(
            "No VisDrone MOT candidate segments are available."
        )

    per_video = (
        all_candidates
        .sort_values(
            "selection_score",
            ascending=False,
        )
        .groupby(
            "video_key",
            as_index=False,
            sort=False,
        )
        .head(1)
        .reset_index(
            drop=True
        )
    )

    if len(per_video) < int(max_videos):
        raise RuntimeError(
            f"Requested {max_videos} sequences, "
            f"but only {len(per_video)} are eligible."
        )

    selected = (
        per_video
        .sample(
            frac=1.0,
            random_state=int(seed),
        )
        .head(
            int(max_videos)
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    selected.insert(
        0,
        "selection_rank",
        np.arange(
            1,
            len(selected) + 1,
        ),
    )

    selected[
        "video_selection_seed"
    ] = int(seed)

    return selected


def parse_visdrone_mot_label_file(
    path: Path,
) -> pd.DataFrame:

    rows = []
    malformed = 0

    columns = [
        "track_id",
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "frame",
        "lost",
        "occluded",
        "generated",
        "label",
        "category_id",
        "truncation",
        "gt_score",
        "line_no",
    ]

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as f:

        for line_no, raw in enumerate(
            f,
            start=1,
        ):
            line = raw.strip()

            if not line:
                continue

            parts = [
                x.strip()
                for x in line.split(",")
            ]

            if len(parts) < 10:
                malformed += 1
                continue

            try:
                frame = int(
                    float(parts[0])
                )

                track_id = int(
                    float(parts[1])
                )

                x, y, w, h = map(
                    float,
                    parts[2:6],
                )

                score = float(
                    parts[6]
                )

                category = int(
                    float(parts[7])
                )

                truncation = int(
                    float(parts[8])
                )

                occlusion = int(
                    float(parts[9])
                )

            except Exception:
                malformed += 1
                continue

            if (
                category
                not in VISDRONE_PERSON_CATEGORY_IDS
            ):
                continue

            if (
                score <= 0
                or track_id <= 0
                or w <= 0
                or h <= 0
            ):
                continue

            rows.append(
                {
                    "track_id": track_id,
                    "xmin": x,
                    "ymin": y,
                    "xmax": x + w,
                    "ymax": y + h,
                    "frame": frame,
                    "lost": 0,
                    "occluded": occlusion,
                    "generated": 0,
                    "label": "person",
                    "category_id": category,
                    "truncation": truncation,
                    "gt_score": score,
                    "line_no": line_no,
                }
            )

    if malformed:
        print(
            f"WARNING: skipped {malformed} "
            f"malformed rows in {path.name}"
        )

    # IMPORTANT FIX:
    # Some VisDrone MOT sequences may contain no valid
    # pedestrian rows. Return an empty table instead of
    # stopping the entire notebook.
    if not rows:
        return pd.DataFrame(
            columns=columns
        )

    return (
        pd.DataFrame(
            rows,
            columns=columns,
        )
        .sort_values(
            [
                "frame",
                "track_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


def _numeric_frame_files(
    sequence_dir: Path,
) -> Dict[int, Path]:

    out = {}

    for p in sequence_dir.iterdir():

        if (
            not p.is_file()
            or p.suffix.lower()
            not in {
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
            }
        ):
            continue

        m = re.search(
            r"(\d+)$",
            p.stem,
        )

        if m:
            out[
                int(
                    m.group(1)
                )
            ] = p

    return out


def discover_visdrone_mot_pairs(
    root: Path,
) -> pd.DataFrame:

    annotation_files = sorted(
        p
        for p in root.rglob(
            "*.txt"
        )
        if p.is_file()
    )

    ann_by_stem = {
        p.stem: p
        for p in annotation_files
    }

    rows = []

    for directory in root.rglob(
        "*"
    ):
        if not directory.is_dir():
            continue

        frame_map = (
            _numeric_frame_files(
                directory
            )
        )

        if not frame_map:
            continue

        seq = directory.name

        label_path = (
            ann_by_stem.get(
                seq
            )
        )

        if label_path is None:
            continue

        first_no = min(
            frame_map
        )

        first_img = cv2.imread(
            str(
                frame_map[
                    first_no
                ]
            )
        )

        if first_img is None:
            continue

        height, width = (
            first_img.shape[:2]
        )

        rows.append(
            {
                "video_key": seq,
                "scenario_key": seq,
                "sequence_dir": str(
                    directory
                ),
                "label_path": str(
                    label_path
                ),
                "label_mode": (
                    "VisDrone2019-MOT official GT"
                ),
                "consistent_ids": True,
                "video_frame_count": len(
                    frame_map
                ),
                "fps": float(
                    VISDRONE_EVAL_FPS
                ),
                "width": int(
                    width
                ),
                "height": int(
                    height
                ),
                "first_frame_number": int(
                    min(
                        frame_map
                    )
                ),
                "last_frame_number": int(
                    max(
                        frame_map
                    )
                ),
            }
        )

    pairs = (
        pd.DataFrame(
            rows
        )
        .drop_duplicates(
            [
                "video_key",
                "sequence_dir",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    if pairs.empty:
        raise RuntimeError(
            "Could not pair VisDrone MOT "
            "sequences with annotation files."
        )

    return pairs


def candidate_segments_for_visdrone_pair(
    pair: pd.Series,
) -> pd.DataFrame:

    labels = (
        parse_visdrone_mot_label_file(
            Path(
                pair[
                    "label_path"
                ]
            )
        )
    )

    # IMPORTANT FIX:
    # Skip sequences without valid pedestrian GT.
    if labels.empty:
        print(
            "Skipping sequence with no "
            "valid pedestrian GT:",
            pair[
                "video_key"
            ],
        )

        return pd.DataFrame()

    frame_map = (
        _numeric_frame_files(
            Path(
                pair[
                    "sequence_dir"
                ]
            )
        )
    )

    frame_numbers = set(
        frame_map
    )

    length = int(
        SEGMENT_LENGTH_FRAMES
    )

    stride = max(
        1,
        int(
            SEGMENT_STRIDE_FRAMES
        ),
    )

    min_frame = max(
        int(
            labels[
                "frame"
            ].min()
        ),
        int(
            pair[
                "first_frame_number"
            ]
        ),
    )

    max_frame = min(
        int(
            labels[
                "frame"
            ].max()
        ),
        int(
            pair[
                "last_frame_number"
            ]
        ),
    )

    rows = []

    for start in range(
        min_frame,
        max_frame
        - length
        + 2,
        stride,
    ):

        end = (
            start
            + length
            - 1
        )

        if not set(
            range(
                start,
                end + 1,
            )
        ).issubset(
            frame_numbers
        ):
            continue

        seg = labels[
            (
                labels[
                    "frame"
                ]
                >= start
            )
            &
            (
                labels[
                    "frame"
                ]
                <= end
            )
        ].copy()

        if seg.empty:
            continue

        counts = (
            seg
            .groupby(
                "frame"
            )
            .size()
            .reindex(
                range(
                    start,
                    end + 1,
                ),
                fill_value=0,
            )
        )

        mean_persons = float(
            counts.mean()
        )

        if (
            mean_persons
            < MIN_MEAN_PERSONS_PER_FRAME
        ):
            continue

        unique_ids = int(
            seg[
                "track_id"
            ].nunique()
        )

        occ = float(
            (
                seg[
                    "occluded"
                ]
                > 0
            ).mean()
        )

        hh = (
            seg[
                "ymax"
            ]
            - seg[
                "ymin"
            ]
        ).clip(
            lower=0
        )

        first_seen = (
            seg
            .groupby(
                "track_id"
            )[
                "frame"
            ]
            .min()
        )

        last_seen = (
            seg
            .groupby(
                "track_id"
            )[
                "frame"
            ]
            .max()
        )

        turnover = int(
            (
                (
                    first_seen
                    > start
                )
                |
                (
                    last_seen
                    < end
                )
            ).sum()
        )

        selection_score = (
            mean_persons
            + 5.0
            * occ
            + 0.20
            * unique_ids
            + 0.15
            * turnover
        )

        rows.append(
            {
                "video_key": pair[
                    "video_key"
                ],
                "scenario_key": pair[
                    "scenario_key"
                ],
                "sequence_dir": pair[
                    "sequence_dir"
                ],
                "label_path": pair[
                    "label_path"
                ],
                "label_mode": pair[
                    "label_mode"
                ],
                "consistent_ids": True,
                "frame_base": 1,
                "start_frame": int(
                    start
                ),
                "end_frame": int(
                    end
                ),
                "length": int(
                    length
                ),
                "video_start_index": 0,
                "fps": float(
                    pair[
                        "fps"
                    ]
                ),
                "width": int(
                    pair[
                        "width"
                    ]
                ),
                "height": int(
                    pair[
                        "height"
                    ]
                ),
                "mean_persons_per_frame": (
                    mean_persons
                ),
                "unique_gt_ids": (
                    unique_ids
                ),
                "occlusion_fraction": (
                    occ
                ),
                "generated_fraction": 0.0,
                "tiny_fraction_height_lt32": float(
                    (
                        hh
                        < 32
                    ).mean()
                ),
                "very_tiny_fraction_height_lt16": float(
                    (
                        hh
                        < 16
                    ).mean()
                ),
                "id_turnover_count": (
                    turnover
                ),
                "selection_score": float(
                    selection_score
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def materialize_visdrone_segment_video(
    segment: pd.Series,
) -> str:

    cache_dir = (
        DATA_CACHE
        / "selected_segment_videos"
    )

    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out = (
        cache_dir
        / (
            safe_slug(
                f"{segment['video_key']}_"
                f"f{int(segment['start_frame']):06d}_"
                f"to_{int(segment['end_frame']):06d}"
            )
            + ".mp4"
        )
    )

    expected = int(
        segment[
            "length"
        ]
    )

    if out.exists():

        cap = cv2.VideoCapture(
            str(
                out
            )
        )

        count = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        cap.release()

        if count == expected:
            return str(
                out
            )

        out.unlink()

    frame_map = (
        _numeric_frame_files(
            Path(
                segment[
                    "sequence_dir"
                ]
            )
        )
    )

    writer = cv2.VideoWriter(
        str(
            out
        ),
        cv2.VideoWriter_fourcc(
            *"mp4v"
        ),
        float(
            segment[
                "fps"
            ]
        ),
        (
            int(
                segment[
                    "width"
                ]
            ),
            int(
                segment[
                    "height"
                ]
            ),
        ),
    )

    if not writer.isOpened():
        raise RuntimeError(
            "Could not create VisDrone "
            f"segment video: {out}"
        )

    try:
        for frame_no in range(
            int(
                segment[
                    "start_frame"
                ]
            ),
            int(
                segment[
                    "end_frame"
                ]
            )
            + 1,
        ):

            path = frame_map.get(
                frame_no
            )

            if path is None:
                raise FileNotFoundError(
                    "Missing VisDrone frame "
                    f"{frame_no}"
                )

            frame = cv2.imread(
                str(
                    path
                )
            )

            if frame is None:
                raise RuntimeError(
                    "Could not read "
                    f"VisDrone frame: {path}"
                )

            writer.write(
                frame
            )

    finally:
        writer.release()

    return str(
        out
    )


# ============================================================
# DISCOVER / BUILD CANDIDATES
# ============================================================

VISDRONE_PAIRS = (
    discover_visdrone_mot_pairs(
        VISDRONE_MOT_ROOT
    )
)

print(
    "Paired VisDrone MOT sequences:",
    len(
        VISDRONE_PAIRS
    ),
)

candidate_tables = []

for _, pair in tqdm(
    VISDRONE_PAIRS.iterrows(),
    total=len(
        VISDRONE_PAIRS
    ),
    desc=(
        "Scanning VisDrone MOT "
        "GT-only candidates"
    ),
):

    table = (
        candidate_segments_for_visdrone_pair(
            pair
        )
    )

    if not table.empty:
        candidate_tables.append(
            table
        )


if not candidate_tables:
    raise RuntimeError(
        "No valid VisDrone MOT "
        "candidate segments were produced."
    )


ALL_CANDIDATES = pd.concat(
    candidate_tables,
    ignore_index=True,
)


print(
    "Valid candidate segments:",
    len(
        ALL_CANDIDATES
    ),
)

print(
    "Eligible source sequences:",
    ALL_CANDIDATES[
        "video_key"
    ].nunique(),
)


# ============================================================
# SEEDED SELECTION
# ============================================================

SELECTED_SEGMENTS = (
    select_seeded_visdrone_segments(
        ALL_CANDIDATES,
        MAX_VIDEOS,
        VIDEO_SELECTION_SEED,
        PREFER_DISTINCT_SCENARIOS,
    )
)


if SELECTED_SEGMENTS.empty:
    raise RuntimeError(
        "VisDrone segment selection "
        "returned zero clips."
    )


SELECTED_SEGMENTS[
    "sequence_name"
] = [
    safe_slug(
        f"visdrone_"
        f"{r.video_key}_"
        f"f{int(r.start_frame):06d}_"
        f"to_{int(r.end_frame):06d}"
    )
    for r
    in SELECTED_SEGMENTS.itertuples()
]


# ============================================================
# MATERIALIZE SELECTED FRAME SEQUENCES AS MP4
# ============================================================

SELECTED_SEGMENTS[
    "video_path"
] = [
    materialize_visdrone_segment_video(
        row
    )
    for _, row in tqdm(
        SELECTED_SEGMENTS.iterrows(),
        total=len(
            SELECTED_SEGMENTS
        ),
        desc=(
            "Materializing selected "
            "VisDrone clips"
        ),
    )
]


SELECTED_SEGMENTS[
    "video_start_index"
] = 0


# ============================================================
# SAVE DATASET AUDIT
# ============================================================

VISDRONE_PAIRS.to_csv(
    DATA_META_DIR
    / "discovered_video_label_pairs.csv",
    index=False,
)

ALL_CANDIDATES.to_csv(
    DATA_META_DIR
    / "all_segment_candidates.csv",
    index=False,
)

SELECTED_SEGMENTS.to_csv(
    DATA_META_DIR
    / "selected_segments.csv",
    index=False,
)


dataset_summary = {
    "dataset": (
        "VisDrone2019-MOT"
    ),
    "split": (
        VISDRONE_MOT_SPLIT
    ),
    "number_of_paired_sequences": int(
        len(
            VISDRONE_PAIRS
        )
    ),
    "number_of_candidate_segments": int(
        len(
            ALL_CANDIDATES
        )
    ),
    "number_of_selected_segments": int(
        len(
            SELECTED_SEGMENTS
        )
    ),
    "selected_unique_videos": int(
        SELECTED_SEGMENTS[
            "video_key"
        ].nunique()
    ),
    "person_category_ids": sorted(
        VISDRONE_PERSON_CATEGORY_IDS
    ),
    "consistent_ids": True,
}


write_json(
    DATA_META_DIR
    / "dataset_summary.json",
    dataset_summary,
)


display(
    SELECTED_SEGMENTS[
        [
            "sequence_name",
            "video_key",
            "start_frame",
            "end_frame",
            "mean_persons_per_frame",
            "unique_gt_ids",
            "occlusion_fraction",
            "id_turnover_count",
        ]
    ]
)


# ============================================================
# SAVE EXACT SEEDED SELECTION
# ============================================================

SELECTED_SEGMENTS.to_csv(
    OUTPUT_ROOT
    / "selected_segments_seeded.csv",
    index=False,
)


write_json(
    OUTPUT_ROOT
    / "video_selection_config.json",
    {
        "dataset": (
            "VisDrone2019-MOT"
        ),
        "split": (
            VISDRONE_MOT_SPLIT
        ),
        "video_selection_seed": int(
            VIDEO_SELECTION_SEED
        ),
        "max_videos": int(
            MAX_VIDEOS
        ),
        "selected_video_keys": [
            str(
                x
            )
            for x in (
                SELECTED_SEGMENTS[
                    "video_key"
                ]
                .tolist()
            )
        ],
    },
)


print(
    f"Selected {len(SELECTED_SEGMENTS)} "
    f"VisDrone MOT sequences "
    f"with VIDEO_SELECTION_SEED="
    f"{VIDEO_SELECTION_SEED}"
)

Paired VisDrone MOT sequences: 17


Scanning VisDrone MOT GT-only candidates:   0%|          | 0/17 [00:00<?, ?it/s]

Skipping sequence with no valid pedestrian GT: uav0000370_00001_v
Valid candidate segments: 13
Eligible source sequences: 10


Materializing selected VisDrone clips:   0%|          | 0/5 [00:00<?, ?it/s]

,sequence_name,video_key,start_frame,end_frame,mean_persons_per_frame,unique_gt_ids,occlusion_fraction,id_turnover_count
0,visdrone_uav0000201_00000_v_f000315_to_000614,uav0000201_00000_v,315,614,1.323333,12,0.226700,12
1,visdrone_uav0000249_00001_v_f000001_to_000300,uav0000249_00001_v,1,300,1.666667,17,0.174000,17
2,visdrone_uav0000073_00600_v_f000001_to_000300,uav0000073_00600_v,1,300,41.996667,88,0.659894,73
3,visdrone_uav0000306_00230_v_f000001_to_000300,uav0000306_00230_v,1,300,3.650000,15,0.364384,15
4,visdrone_uav0000161_00000_v_f000001_to_000300,uav0000161_00000_v,1,300,8.106667,15,0.171875,11


Selected 5 VisDrone MOT sequences with VIDEO_SELECTION_SEED=100


## 8. BPD-YOLOn/L-FPN DySample compatibility

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class DySample(nn.Module):
    """Project-compatible content-aware point-sampling upsampler."""
    def __init__(self, channels: Optional[int] = None, scale: int = 2, max_offset: float = 0.25):
        super().__init__()
        self.channels = int(channels) if channels is not None else None
        self.scale = int(scale)
        self.max_offset = float(max_offset)
        self._zero_initialized = False
        if self.channels is None:
            self.offset = nn.LazyConv2d(2, kernel_size=1, stride=1, padding=0)
        else:
            self.offset = nn.Conv2d(self.channels, 2, kernel_size=1, stride=1, padding=0)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)
            self._zero_initialized = True

    def _materialize_legacy_lazy_offset(self, x):
        if isinstance(self.offset, nn.LazyConv2d) and not getattr(self, "_zero_initialized", False):
            _ = self.offset(x)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)
            self._zero_initialized = True

    def forward(self, x):
        channels = getattr(self, "channels", None)
        if channels is not None and x.shape[1] != int(channels):
            raise RuntimeError(f"DySample expected {int(channels)} channels but received {x.shape[1]}")
        self._materialize_legacy_lazy_offset(x)
        b, _, h, w = x.shape
        scale = int(getattr(self, "scale", 2))
        max_offset = float(getattr(self, "max_offset", 0.25))
        oh, ow = h * scale, w * scale
        raw = self.offset(x)
        raw = F.interpolate(raw, size=(oh, ow), mode="bilinear", align_corners=False)
        raw = torch.tanh(raw) * max_offset
        ys = ((torch.arange(oh, device=x.device, dtype=x.dtype) + 0.5) / oh) * 2.0 - 1.0
        xs = ((torch.arange(ow, device=x.device, dtype=x.dtype) + 0.5) / ow) * 2.0 - 1.0
        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        base = torch.stack((xx, yy), dim=-1).unsqueeze(0).expand(b, -1, -1, -1)
        dx = raw[:, 0] * (2.0 / max(w, 1))
        dy = raw[:, 1] * (2.0 / max(h, 1))
        grid = base + torch.stack((dx, dy), dim=-1)
        return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def register_bpd_compatibility():
    import ultralytics.nn.tasks as ultralytics_tasks
    setattr(sys.modules["__main__"], "DySample", DySample)
    setattr(ultralytics_tasks, "DySample", DySample)
    try:
        import ultralytics.nn.modules as ultralytics_modules
        setattr(ultralytics_modules, "DySample", DySample)
    except Exception:
        pass
    try:
        torch.serialization.add_safe_globals([DySample])
    except Exception:
        pass


register_bpd_compatibility()
print("BPD DySample compatibility: REGISTERED")


BPD DySample compatibility: REGISTERED


## 9. Pinned official RT-DETR source
Required for the final `.pth` checkpoint. If the Step-4 `.onnx` export is supplied,
ONNX Runtime is used directly instead.

In [ ]:
RTDETR_LOCAL_ROOT = TOOL_CACHE / "rtdetr"
RTDETR_REPO_ROOT = RTDETR_LOCAL_ROOT / "RT-DETR"
RTDETR_ROOT = RTDETR_REPO_ROOT / "rtdetrv2_pytorch"
RTDETR_CONFIG_DIR = RTDETR_ROOT / "configs" / "custom"
RTDETR_CONFIG_PATH = RTDETR_CONFIG_DIR / "rtdetr_r18_person_step5.yml"


def prepare_rtdetr_repository() -> None:
    if not RTDETR_REPO_ROOT.exists():
        RTDETR_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["git", "clone", "--filter=blob:none", RTDETR_REPO_URL, str(RTDETR_REPO_ROOT)])
    subprocess.check_call(["git", "-C", str(RTDETR_REPO_ROOT), "fetch", "--all", "--tags", "--prune"])
    subprocess.check_call(["git", "-C", str(RTDETR_REPO_ROOT), "checkout", "--force", RTDETR_REPO_COMMIT])
    if not RTDETR_ROOT.exists():
        raise FileNotFoundError(RTDETR_ROOT)
    RTDETR_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    config_text = f'''__include__:
  - ../rtdetr/rtdetr_r18vd_6x_coco.yml

num_classes: 1
remap_mscoco_category: False

eval_spatial_size: [{IMAGE_SIZE}, {IMAGE_SIZE}]

PResNet:
  pretrained: False
'''
    RTDETR_CONFIG_PATH.write_text(config_text, encoding="utf-8")
    commit = subprocess.check_output(["git", "-C", str(RTDETR_REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    write_json(AUDIT_DIR / "rtdetr_source.json", {
        "repo": RTDETR_REPO_URL, "requested_commit": RTDETR_REPO_COMMIT,
        "resolved_commit": commit, "config": str(RTDETR_CONFIG_PATH),
    })


if MODEL_PATHS["RT-DETR-R18"].suffix.lower() == ".pth":
    prepare_rtdetr_repository()
    print("RT-DETR repository/config: READY")
else:
    print("RT-DETR ONNX source selected; PyTorch repository setup deferred/not needed.")


RT-DETR ONNX source selected; PyTorch repository setup deferred/not needed.


## 10. Unified detector backends

In [ ]:
from ultralytics import YOLO


@dataclass
class DetectionBatch:
    boxes: np.ndarray
    scores: np.ndarray
    classes: np.ndarray
    preprocess_ms: float = 0.0
    inference_ms: float = 0.0
    postprocess_ms: float = 0.0
    total_ms: float = 0.0

    def __len__(self):
        return int(len(self.boxes))

    @staticmethod
    def empty(total_ms: float = 0.0):
        return DetectionBatch(
            np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32),
            np.zeros((0,), dtype=np.float32), total_ms=float(total_ms)
        )


class UltralyticsDetector:
    def __init__(self, model_path: Path, model_name: str, is_bpd: bool = False):
        self.model_path = Path(model_path)
        self.model_name = model_name
        if is_bpd and self.model_path.suffix.lower() == ".pt":
            register_bpd_compatibility()
        self.model = YOLO(str(self.model_path))
        try:
            self.model.names = {0: "person"}
        except Exception:
            pass

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        cuda_sync(); total_start = time.perf_counter()
        results = self.model.predict(
            source=frame, imgsz=IMAGE_SIZE, conf=detector_conf_threshold(self.model_name),
            iou=DETECTOR_NMS_IOU, max_det=MAX_DETECTIONS, classes=[0],
            device=DEVICE_ID,
            verbose=False,
        )
        cuda_sync(); total_ms = (time.perf_counter() - total_start) * 1000.0
        if not results:
            return DetectionBatch.empty(total_ms)
        result = results[0]
        speed = getattr(result, "speed", {}) or {}
        if result.boxes is None or len(result.boxes) == 0:
            return DetectionBatch(
                np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.float32),
                float(speed.get("preprocess", 0.0) or 0.0), float(speed.get("inference", 0.0) or 0.0),
                float(speed.get("postprocess", 0.0) or 0.0), total_ms
            )
        boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
        scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
        mask = scores >= detector_conf_threshold(self.model_name)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        return DetectionBatch(
            boxes, scores, np.zeros((len(boxes),), np.float32),
            float(speed.get("preprocess", 0.0) or 0.0), float(speed.get("inference", 0.0) or 0.0),
            float(speed.get("postprocess", 0.0) or 0.0), total_ms
        )


class RTDETRPyTorchDetector:
    def __init__(self, checkpoint_path: Path):
        self.checkpoint_path = Path(checkpoint_path)
        self._build()

    @staticmethod
    def _extract_state_dict(checkpoint):
        if not isinstance(checkpoint, dict):
            raise RuntimeError("Unexpected RT-DETR checkpoint format")
        ema = checkpoint.get("ema")
        if ema is not None:
            if isinstance(ema, dict) and "module" in ema and isinstance(ema["module"], dict):
                return ema["module"]
            if isinstance(ema, dict):
                return ema
            if hasattr(ema, "state_dict"):
                return ema.state_dict()
        model_state = checkpoint.get("model")
        if model_state is not None:
            if isinstance(model_state, dict):
                return model_state
            if hasattr(model_state, "state_dict"):
                return model_state.state_dict()
        raise RuntimeError("Could not find RT-DETR weights under 'ema' or 'model'")

    def _build(self):
        prepare_rtdetr_repository()
        if str(RTDETR_ROOT) not in sys.path:
            sys.path.insert(0, str(RTDETR_ROOT))
        importlib.invalidate_caches()
        old_cwd = Path.cwd()
        try:
            os.chdir(RTDETR_ROOT)
            from src.core import YAMLConfig
            cfg = YAMLConfig(str(RTDETR_CONFIG_PATH), device=f"cuda:{DEVICE_ID}", use_amp=False)
        finally:
            os.chdir(old_cwd)
        checkpoint = torch.load(self.checkpoint_path, map_location="cpu", weights_only=False)
        state = self._extract_state_dict(checkpoint)
        cfg.model.load_state_dict(state, strict=True)
        self.model = cfg.model.deploy().cuda(DEVICE_ID).eval()
        self.postprocessor = cfg.postprocessor.deploy()
        del checkpoint, state
        gc.collect(); torch.cuda.empty_cache()

    def _preprocess(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        array = resized.astype(np.float32) / 255.0
        return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).contiguous().cuda(DEVICE_ID, non_blocking=True)

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        h, w = frame.shape[:2]
        total_start = time.perf_counter()
        pre_start = time.perf_counter()
        tensor = self._preprocess(frame)
        original_size = torch.tensor([[w, h]], dtype=torch.float32, device=f"cuda:{DEVICE_ID}")
        cuda_sync(); preprocess_ms = (time.perf_counter() - pre_start) * 1000.0
        cuda_sync(); infer_start = time.perf_counter()
        with torch.inference_mode():
            ctx = torch.autocast("cuda", dtype=torch.float16) if USE_FP16_CHECKPOINT_INFERENCE else contextlib.nullcontext()
            with ctx:
                outputs = self.model(tensor)
        cuda_sync(); inference_ms = (time.perf_counter() - infer_start) * 1000.0
        post_start = time.perf_counter()
        with torch.inference_mode():
            labels, boxes, scores = self.postprocessor(outputs, original_size)
        cuda_sync()
        labels = labels[0].detach().cpu().numpy()
        boxes = boxes[0].detach().cpu().numpy().astype(np.float32)
        scores = scores[0].detach().cpu().numpy().astype(np.float32)
        mask = (scores >= detector_conf_threshold("RT-DETR-R18")) & (labels.astype(np.int64) == 0)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        postprocess_ms = (time.perf_counter() - post_start) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0
        return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32),
                              preprocess_ms, inference_ms, postprocess_ms, total_ms)


def ort_numpy_dtype(input_type: str):
    t = str(input_type).lower()
    if "int64" in t: return np.int64
    if "int32" in t: return np.int32
    if "float16" in t: return np.float16
    return np.float32


class RTDETRONNXDetector:
    def __init__(self, onnx_path: Path):
        import onnxruntime as ort
        available = ort.get_available_providers()
        providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in available else []) + ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.inputs = {x.name: x for x in self.session.get_inputs()}
        self.output_names = [x.name for x in self.session.get_outputs()]
        if not {"images", "orig_target_sizes"}.issubset(self.inputs):
            raise RuntimeError(f"Unexpected RT-DETR ONNX inputs: {list(self.inputs)}")
        if not {"labels", "boxes", "scores"}.issubset(self.output_names):
            raise RuntimeError(f"Unexpected RT-DETR ONNX outputs: {self.output_names}")
        print("RT-DETR ONNX providers:", self.session.get_providers())

    def predict(self, frame: np.ndarray) -> DetectionBatch:
        h, w = frame.shape[:2]; total_start = time.perf_counter(); pre = time.perf_counter()
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        images = (resized.astype(np.float32).transpose(2, 0, 1)[None] / 255.0).astype(ort_numpy_dtype(self.inputs["images"].type), copy=False)
        sizes = np.asarray([[w, h]], dtype=ort_numpy_dtype(self.inputs["orig_target_sizes"].type))
        preprocess_ms = (time.perf_counter() - pre) * 1000.0
        inf = time.perf_counter(); outputs = self.session.run(self.output_names, {"images": images, "orig_target_sizes": sizes})
        inference_ms = (time.perf_counter() - inf) * 1000.0; post = time.perf_counter()
        out = dict(zip(self.output_names, outputs))
        labels, boxes, scores = np.asarray(out["labels"])[0], np.asarray(out["boxes"])[0].astype(np.float32), np.asarray(out["scores"])[0].astype(np.float32)
        mask = (scores >= detector_conf_threshold("RT-DETR-R18")) & (labels.astype(np.int64) == 0)
        boxes, scores = boxes[mask], scores[mask]
        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]; boxes, scores = boxes[order], scores[order]
        postprocess_ms = (time.perf_counter() - post) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0
        return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32), preprocess_ms, inference_ms, postprocess_ms, total_ms)



def trt_dtype_to_torch(dtype):
    np_dtype = np.dtype(trt.nptype(dtype))

    mapping = {
        np.dtype(np.float32): torch.float32,
        np.dtype(np.float16): torch.float16,
        np.dtype(np.int64): torch.int64,
        np.dtype(np.int32): torch.int32,
        np.dtype(np.int8): torch.int8,
        np.dtype(np.bool_): torch.bool,
    }

    if np_dtype not in mapping:
        raise TypeError(f"Unsupported TensorRT dtype: {dtype} / {np_dtype}")

    return mapping[np_dtype]


class RTDETRTensorRTDetector:
    def __init__(self, engine_path: Path):
        self.engine_path = Path(engine_path)
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)

        engine_bytes = self.engine_path.read_bytes()
        self.engine = self.runtime.deserialize_cuda_engine(engine_bytes)

        if self.engine is None:
            raise RuntimeError(
                "Could not deserialize the RT-DETR TensorRT engine. "
                "TensorRT engines are hardware/runtime specific. "
                f"Engine={self.engine_path}, TensorRT={trt.__version__}, "
                f"GPU={torch.cuda.get_device_name(DEVICE_ID)}"
            )

        self.context = self.engine.create_execution_context()

        if self.context is None:
            raise RuntimeError("Could not create RT-DETR TensorRT execution context.")

        self.stream = torch.cuda.Stream(device=DEVICE_ID)
        self.input_names = []
        self.output_names = []

        for index in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(index)
            mode = self.engine.get_tensor_mode(name)

            if mode == trt.TensorIOMode.INPUT:
                self.input_names.append(name)
            else:
                self.output_names.append(name)

        if not {"images", "orig_target_sizes"}.issubset(self.input_names):
            raise RuntimeError(f"Unexpected RT-DETR TensorRT inputs: {self.input_names}")

        if not {"labels", "boxes", "scores"}.issubset(self.output_names):
            raise RuntimeError(f"Unexpected RT-DETR TensorRT outputs: {self.output_names}")

        self.buffers = {}
        self._initialize_buffers()

        print("RT-DETR TensorRT engine: LOADED")
        print("Inputs :", self.input_names)
        print("Outputs:", self.output_names)

    def _initialize_buffers(self):
        input_shapes = {
            "images": (1, 3, IMAGE_SIZE, IMAGE_SIZE),
            "orig_target_sizes": (1, 2),
        }

        for name in self.input_names:
            engine_shape = tuple(
                int(v) for v in self.engine.get_tensor_shape(name)
            )

            if any(v < 0 for v in engine_shape):
                shape = input_shapes.get(name)

                if shape is None:
                    raise RuntimeError(f"No static input-shape rule for: {name}")

                ok = self.context.set_input_shape(name, shape)

                if ok is False:
                    raise RuntimeError(f"TensorRT rejected {name} shape {shape}")

        for index in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(index)
            dtype = self.engine.get_tensor_dtype(name)
            torch_dtype = trt_dtype_to_torch(dtype)
            shape = tuple(
                int(v) for v in self.context.get_tensor_shape(name)
            )

            if any(v < 0 for v in shape):
                raise RuntimeError(
                    f"TensorRT tensor still has dynamic shape: {name} -> {shape}"
                )

            tensor = torch.empty(
                shape,
                dtype=torch_dtype,
                device=f"cuda:{DEVICE_ID}",
            )

            self.buffers[name] = tensor
            self.context.set_tensor_address(
                name,
                int(tensor.data_ptr()),
            )

    def _prepare_inputs(self, frame):
        height, width = frame.shape[:2]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(
            rgb,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_LINEAR,
        )

        image_np = (
            resized.astype(np.float32)
            .transpose(2, 0, 1)[None]
            / 255.0
        )

        image_tensor = torch.from_numpy(image_np).to(
            device=f"cuda:{DEVICE_ID}",
            dtype=self.buffers["images"].dtype,
        )

        size_tensor = torch.tensor(
            [[width, height]],
            device=f"cuda:{DEVICE_ID}",
            dtype=self.buffers["orig_target_sizes"].dtype,
        )

        self.buffers["images"].copy_(image_tensor)
        self.buffers["orig_target_sizes"].copy_(size_tensor)

    def predict(self, frame):
        total_start = time.perf_counter()

        pre_start = time.perf_counter()
        self._prepare_inputs(frame)
        cuda_sync()
        preprocess_ms = (time.perf_counter() - pre_start) * 1000.0

        cuda_sync()
        infer_start = time.perf_counter()

        ok = self.context.execute_async_v3(
            stream_handle=int(self.stream.cuda_stream)
        )

        if not ok:
            raise RuntimeError("RT-DETR TensorRT execute_async_v3 returned False.")

        self.stream.synchronize()
        inference_ms = (time.perf_counter() - infer_start) * 1000.0

        post_start = time.perf_counter()

        labels = self.buffers["labels"][0].detach().cpu().numpy()
        boxes = (
            self.buffers["boxes"][0]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        scores = (
            self.buffers["scores"][0]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        mask = (
            (scores >= detector_conf_threshold("RT-DETR-R18"))
            & (labels.astype(np.int64) == 0)
        )

        boxes = boxes[mask]
        scores = scores[mask]

        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]
            boxes = boxes[order]
            scores = scores[order]

        postprocess_ms = (time.perf_counter() - post_start) * 1000.0
        total_ms = (time.perf_counter() - total_start) * 1000.0

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=np.zeros((len(boxes),), dtype=np.float32),
            preprocess_ms=preprocess_ms,
            inference_ms=inference_ms,
            postprocess_ms=postprocess_ms,
            total_ms=total_ms,
        )


def build_detector(model_name: str):
    path = MODEL_PATHS[model_name]
    suffix = path.suffix.lower()

    if model_name == "RT-DETR-R18":
        if suffix == ".pth":
            return RTDETRPyTorchDetector(path)
        if suffix == ".onnx":
            return RTDETRONNXDetector(path)
        if suffix == ".engine":
            return RTDETRTensorRTDetector(path)
        raise ValueError("RT-DETR Step-5 supports .pth, .onnx, or .engine.")

    if model_name == "YOLO26s":
        if suffix not in {".pt", ".onnx", ".engine"}:
            raise ValueError(f"Unsupported YOLO26s format: {suffix}")
        return UltralyticsDetector(path, model_name, False)

    if model_name == "BPD-YOLOn/L-FPN":
        if suffix not in {".pt", ".onnx", ".engine"}:
            raise ValueError(f"Unsupported BPD format: {suffix}")
        return UltralyticsDetector(path, model_name, suffix == ".pt")

    raise KeyError(model_name)


print("Unified detector backends: READY")


Unified detector backends: READY


## 11. Common ByteTrack / BoT-SORT adapters
Both receive the exact same detector boxes/scores. BoT-SORT uses sparse-optical-flow
camera-motion compensation; ReID is disabled in the primary controlled comparison.

In [ ]:
from dataclasses import dataclass
from types import SimpleNamespace
from typing import Any, Dict

import numpy as np
import torch

from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.bot_sort import BOTSORT


@dataclass
class TrackBatch:
    boxes: np.ndarray
    ids: np.ndarray
    scores: np.ndarray

    def __len__(self):
        return int(
            len(self.ids)
        )

    @staticmethod
    def empty():
        return TrackBatch(
            boxes=np.zeros(
                (0, 4),
                dtype=np.float32,
            ),
            ids=np.zeros(
                (0,),
                dtype=np.int32,
            ),
            scores=np.zeros(
                (0,),
                dtype=np.float32,
            ),
        )


class CommonTracker:
    """
    Unified ByteTrack / BoT-SORT adapter.

    IMPORTANT:
    - No auxiliary neural ReID model is used.
    - BoT-SORT with_reid is always False.
    - model='auto' is only a required placeholder for the
      current Ultralytics BOTSORT constructor.
    - Lost tracks are retained for TRACK_LOST_GRACE_SECONDS.
    """

    def __init__(
        self,
        model_name: str,
        tracker_name: str,
        frame_rate: float,
    ):
        self.model_name = str(
            model_name
        )

        self.tracker_name = str(
            tracker_name
        )

        self.frame_rate = max(
            1.0,
            float(
                frame_rate
            ),
        )

        # Convert the requested lost-track grace
        # from seconds to source-video frames.
        self.track_buffer_frames = max(
            1,
            int(
                round(
                    self.frame_rate
                    * TRACK_LOST_GRACE_SECONDS
                )
            ),
        )

        self.effective_settings = None
        self.tracker = None

        self._build()

    def _settings(
        self,
    ) -> Dict[str, Any]:

        if (
            self.model_name
            not in TRACKER_MODEL_THRESHOLDS
        ):
            raise KeyError(
                "No tracker threshold profile "
                f"for detector: {self.model_name}"
            )

        # -----------------------------------------------------
        # ByteTrack
        # -----------------------------------------------------
        if (
            self.tracker_name
            == "ByteTrack"
        ):
            settings = dict(
                BYTETRACK_SETTINGS
            )

        # -----------------------------------------------------
        # BoT-SORT
        # -----------------------------------------------------
        elif (
            self.tracker_name
            == "BoT-SORT"
        ):
            settings = dict(
                BOTSORT_SETTINGS
            )

            # -------------------------------------------------
            # ReID MUST remain disabled.
            # -------------------------------------------------
            settings[
                "with_reid"
            ] = False

            # Current Ultralytics BOTSORT constructor accesses
            # args.model even when with_reid=False.
            #
            # This is ONLY a placeholder.
            # It does NOT load a ReID model because
            # with_reid=False.
            settings[
                "model"
            ] = "auto"

            # Same compatibility placeholder.
            settings[
                "device"
            ] = "cuda"

        else:
            raise ValueError(
                "Unsupported tracker: "
                f"{self.tracker_name}"
            )

        # -----------------------------------------------------
        # Detector-specific tracking thresholds
        # -----------------------------------------------------
        settings.update(
            TRACKER_MODEL_THRESHOLDS[
                self.model_name
            ]
        )

        # -----------------------------------------------------
        # Exact 2-second lost-track retention
        # -----------------------------------------------------
        settings[
            "track_buffer"
        ] = int(
            self.track_buffer_frames
        )

        return settings

    def _build(
        self,
    ):
        settings = (
            self._settings()
        )

        # -----------------------------------------------------
        # Select tracker class
        # -----------------------------------------------------
        if (
            self.tracker_name
            == "ByteTrack"
        ):
            tracker_cls = (
                BYTETracker
            )

            tracker_type = (
                "bytetrack"
            )

        elif (
            self.tracker_name
            == "BoT-SORT"
        ):
            tracker_cls = (
                BOTSORT
            )

            tracker_type = (
                "botsort"
            )

        else:
            raise ValueError(
                "Unsupported tracker: "
                f"{self.tracker_name}"
            )

        # -----------------------------------------------------
        # Build Ultralytics tracker arguments
        # -----------------------------------------------------
        args = SimpleNamespace(
            tracker_type=tracker_type,
            **settings,
        )

        self.effective_settings = {
            **settings,

            "tracker_type": (
                tracker_type
            ),

            "model_name": (
                self.model_name
            ),

            "frame_rate": (
                self.frame_rate
            ),

            "track_lost_grace_seconds": (
                self.track_buffer_frames
                / self.frame_rate
            ),

            # Explicit audit flags
            "internal_reid_active": False,
            "external_reid_active": False,
        }

        # -----------------------------------------------------
        # Build tracker
        # -----------------------------------------------------
        #
        # Some Ultralytics versions accept frame_rate,
        # while the current BOTSORT version accepts only args.
        # The fallback handles both.
        # -----------------------------------------------------
        try:
            self.tracker = tracker_cls(
                args,
                frame_rate=self.frame_rate,
            )

        except TypeError:
            self.tracker = tracker_cls(
                args
            )

        # -----------------------------------------------------
        # Verify actual lost-track buffer
        # -----------------------------------------------------
        actual_buffer = getattr(
            self.tracker,
            "max_frames_lost",
            getattr(
                self.tracker,
                "max_time_lost",
                self.track_buffer_frames,
            ),
        )

        try:
            actual_buffer = int(
                actual_buffer
            )

        except Exception:
            actual_buffer = int(
                self.track_buffer_frames
            )

        self.effective_settings[
            "actual_tracker_buffer_frames"
        ] = int(
            actual_buffer
        )

        self.effective_settings[
            "actual_tracker_buffer_seconds"
        ] = float(
            actual_buffer
            / self.frame_rate
        )

        # -----------------------------------------------------
        # Final ReID audit
        # -----------------------------------------------------
        self.effective_settings[
            "internal_reid_active"
        ] = False

        self.effective_settings[
            "external_reid_active"
        ] = False

    def reset(
        self,
    ):
        self._build()

    def audit_dict(
        self,
    ) -> Dict[str, Any]:

        return dict(
            self.effective_settings
            or {}
        )

    def update(
        self,
        detections,
        frame: np.ndarray,
    ) -> TrackBatch:
        """
        Update the tracker using detector output.

        Expected detector object:
            detections.boxes  -> Nx4 xyxy
            detections.scores -> N confidence values

        DetectionBatch is intentionally not used as a type
        annotation to avoid notebook execution-order issues.
        """

        h, w = (
            frame.shape[:2]
        )

        # -----------------------------------------------------
        # Empty detection case
        # -----------------------------------------------------
        if (
            detections is None
            or len(
                detections
            ) == 0
        ):
            data = torch.empty(
                (0, 6),
                dtype=torch.float32,
            )

        # -----------------------------------------------------
        # Convert detections to Ultralytics Boxes
        # -----------------------------------------------------
        else:
            boxes_np = np.asarray(
                detections.boxes,
                dtype=np.float32,
            ).reshape(
                -1,
                4,
            )

            scores_np = np.asarray(
                detections.scores,
                dtype=np.float32,
            ).reshape(
                -1
            )

            if (
                len(
                    boxes_np
                )
                != len(
                    scores_np
                )
            ):
                raise RuntimeError(
                    "Detection boxes/scores "
                    "length mismatch: "
                    f"{len(boxes_np)} vs "
                    f"{len(scores_np)}"
                )

            # person-only experiment:
            # all detections use class 0.
            cls_col = np.zeros(
                (
                    len(
                        boxes_np
                    ),
                    1,
                ),
                dtype=np.float32,
            )

            data_np = np.concatenate(
                [
                    boxes_np,

                    scores_np[
                        :,
                        None,
                    ],

                    cls_col,
                ],
                axis=1,
            )

            data = (
                torch.from_numpy(
                    data_np
                )
                .float()
            )

        # Ultralytics expected format:
        #
        # x1, y1, x2, y2, confidence, class
        results = Boxes(
            data,
            (
                h,
                w,
            ),
        )

        # -----------------------------------------------------
        # Run tracker
        # -----------------------------------------------------
        tracks = (
            self.tracker.update(
                results,
                frame,
            )
        )

        # -----------------------------------------------------
        # No active tracks
        # -----------------------------------------------------
        if (
            tracks is None
            or len(
                tracks
            ) == 0
        ):
            return (
                TrackBatch.empty()
            )

        tracks = np.asarray(
            tracks,
            dtype=np.float32,
        )

        if (
            tracks.ndim
            == 1
        ):
            tracks = tracks[
                None,
                :
            ]

        if (
            tracks.ndim != 2
            or tracks.shape[1] < 6
        ):
            raise RuntimeError(
                f"Unexpected "
                f"{self.tracker_name} "
                f"output shape: "
                f"{tracks.shape}"
            )

        # -----------------------------------------------------
        # Ultralytics tracker output
        #
        # [x1, y1, x2, y2, track_id, score, ...]
        # -----------------------------------------------------
        track_boxes = (
            tracks[
                :,
                :4,
            ]
            .astype(
                np.float32
            )
        )

        track_ids = (
            tracks[
                :,
                4,
            ]
            .astype(
                np.int32
            )
        )

        track_scores = (
            tracks[
                :,
                5,
            ]
            .astype(
                np.float32
            )
        )

        return TrackBatch(
            boxes=track_boxes,
            ids=track_ids,
            scores=track_scores,
        )


# ============================================================
# AUDIT
# ============================================================

print(
    "Tracker adapters: READY"
)

print(
    "ID-stable lost-track grace:",
    f"{TRACK_LOST_GRACE_SECONDS:.1f} seconds",
)

print(
    "ByteTrack ReID: NOT USED"
)

print(
    "BoT-SORT neural ReID: DISABLED"
)

print(
    "External ReID neural model: DISABLED"
)

Tracker adapters: READY
ID-stable lost-track grace: 2.0 seconds
ByteTrack ReID: NOT USED
BoT-SORT neural ReID: DISABLED
External ReID neural model: DISABLED


## 11B. ReID policy

External ReID and Long-Term Global-ID recovery are disabled.

In [ ]:
REID_MODEL_USED = False
print("External ReID neural model: DISABLED")
print("TrackEval identity source  : raw tracker IDs")

External ReID neural model: DISABLED
TrackEval identity source  : raw tracker IDs


## 12. Detection/MOT metrics, video, and segment utilities

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import cv2
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment


def xyxy_to_xywh(
    boxes: np.ndarray,
) -> np.ndarray:
    boxes = np.asarray(
        boxes,
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    out = boxes.copy()

    if len(out):
        out[:, 2] = (
            boxes[:, 2]
            - boxes[:, 0]
        )

        out[:, 3] = (
            boxes[:, 3]
            - boxes[:, 1]
        )

    return out


def box_iou_matrix(
    a: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    a = np.asarray(
        a,
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    b = np.asarray(
        b,
        dtype=np.float32,
    ).reshape(
        -1,
        4,
    )

    if (
        len(a) == 0
        or len(b) == 0
    ):
        return np.zeros(
            (
                len(a),
                len(b),
            ),
            dtype=np.float32,
        )

    lt = np.maximum(
        a[:, None, :2],
        b[None, :, :2],
    )

    rb = np.minimum(
        a[:, None, 2:],
        b[None, :, 2:],
    )

    wh = np.clip(
        rb - lt,
        0,
        None,
    )

    inter = (
        wh[..., 0]
        * wh[..., 1]
    )

    area_a = (
        np.clip(
            a[:, 2] - a[:, 0],
            0,
            None,
        )
        *
        np.clip(
            a[:, 3] - a[:, 1],
            0,
            None,
        )
    )

    area_b = (
        np.clip(
            b[:, 2] - b[:, 0],
            0,
            None,
        )
        *
        np.clip(
            b[:, 3] - b[:, 1],
            0,
            None,
        )
    )

    union = (
        area_a[:, None]
        + area_b[None, :]
        - inter
    )

    return (
        inter
        / np.clip(
            union,
            1e-9,
            None,
        )
    )


def match_boxes(
    gt_boxes: np.ndarray,
    pred_boxes: np.ndarray,
    iou_threshold: float,
):
    ious = box_iou_matrix(
        gt_boxes,
        pred_boxes,
    )

    if not ious.size:
        return (
            [],
            set(),
            set(),
        )

    gt_indices, pred_indices = (
        linear_sum_assignment(
            -ious
        )
    )

    matches = []
    matched_gt = set()
    matched_pred = set()

    for gt_idx, pred_idx in zip(
        gt_indices,
        pred_indices,
    ):
        iou = float(
            ious[
                gt_idx,
                pred_idx,
            ]
        )

        if (
            iou
            >= float(
                iou_threshold
            )
        ):
            matches.append(
                (
                    int(gt_idx),
                    int(pred_idx),
                    iou,
                )
            )

            matched_gt.add(
                int(gt_idx)
            )

            matched_pred.add(
                int(pred_idx)
            )

    return (
        matches,
        matched_gt,
        matched_pred,
    )


def detection_operating_metrics(
    gt_df: pd.DataFrame,
    detections_by_original_frame: Dict[int, Any],
) -> Dict[str, Any]:
    """
    Compute detector operating-point metrics.

    This function intentionally does NOT use DetectionBatch
    as a type annotation, so it remains safe regardless of
    notebook cell execution order.

    Expected detection object attributes:
        det.boxes
        det.scores
    """

    tp = 0
    fp = 0
    fn = 0

    occ_total = 0
    occ_hit = 0

    vis_total = 0
    vis_hit = 0

    bins = {
        "h_lt16": {
            "total": 0,
            "hit": 0,
        },
        "h_16_31": {
            "total": 0,
            "hit": 0,
        },
        "h_32_95": {
            "total": 0,
            "hit": 0,
        },
        "h_ge96": {
            "total": 0,
            "hit": 0,
        },
    }

    for (
        original_frame,
        frame_gt,
    ) in gt_df.groupby(
        "frame"
    ):
        gt_boxes = (
            frame_gt[
                [
                    "xmin",
                    "ymin",
                    "xmax",
                    "ymax",
                ]
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        det = (
            detections_by_original_frame
            .get(
                int(
                    original_frame
                )
            )
        )

        if det is None:
            pred_boxes = np.zeros(
                (
                    0,
                    4,
                ),
                dtype=np.float32,
            )

        else:
            pred_boxes = np.asarray(
                det.boxes,
                dtype=np.float32,
            ).reshape(
                -1,
                4,
            )

        (
            matches,
            matched_gt,
            matched_pred,
        ) = match_boxes(
            gt_boxes,
            pred_boxes,
            DETECTION_EVAL_IOU,
        )

        tp += len(
            matches
        )

        fp += (
            len(
                pred_boxes
            )
            - len(
                matched_pred
            )
        )

        fn += (
            len(
                gt_boxes
            )
            - len(
                matched_gt
            )
        )

        reset_gt = (
            frame_gt
            .reset_index(
                drop=True
            )
        )

        for (
            idx,
            row,
        ) in reset_gt.iterrows():
            hit = (
                int(idx)
                in matched_gt
            )

            # VisDrone occlusion:
            # 0 = none
            # 1/2 = partial/heavy
            if (
                int(
                    row[
                        "occluded"
                    ]
                )
                > 0
            ):
                occ_total += 1
                occ_hit += int(
                    hit
                )

            else:
                vis_total += 1
                vis_hit += int(
                    hit
                )

            height = float(
                row["ymax"]
                - row["ymin"]
            )

            if height < 16:
                key = "h_lt16"

            elif height < 32:
                key = "h_16_31"

            elif height < 96:
                key = "h_32_95"

            else:
                key = "h_ge96"

            bins[
                key
            ][
                "total"
            ] += 1

            bins[
                key
            ][
                "hit"
            ] += int(
                hit
            )

    precision = (
        tp
        / max(
            tp + fp,
            1,
        )
    )

    recall = (
        tp
        / max(
            tp + fn,
            1,
        )
    )

    f1 = (
        2.0
        * precision
        * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    out = {
        "tp": int(
            tp
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "visible_recall": float(
            vis_hit
            / max(
                vis_total,
                1,
            )
        ),
        "occluded_recall": float(
            occ_hit
            / max(
                occ_total,
                1,
            )
        ),
        "visible_gt": int(
            vis_total
        ),
        "occluded_gt": int(
            occ_total
        ),
    }

    for (
        key,
        value,
    ) in bins.items():
        out[
            f"{key}_gt"
        ] = int(
            value[
                "total"
            ]
        )

        out[
            f"{key}_recall"
        ] = float(
            value[
                "hit"
            ]
            / max(
                value[
                    "total"
                ],
                1,
            )
        )

    return out


def load_segment_gt(
    segment: pd.Series,
) -> pd.DataFrame:
    labels = (
        parse_visdrone_mot_label_file(
            Path(
                segment[
                    "label_path"
                ]
            )
        )
    )

    start_frame = int(
        segment[
            "start_frame"
        ]
    )

    end_frame = int(
        segment[
            "end_frame"
        ]
    )

    return labels[
        (
            labels[
                "frame"
            ]
            >= start_frame
        )
        &
        (
            labels[
                "frame"
            ]
            <= end_frame
        )
    ].copy()


def frame_gt_dict(
    gt_df: pd.DataFrame,
):
    return {
        int(
            frame_number
        ): group.copy()
        for (
            frame_number,
            group,
        ) in gt_df.groupby(
            "frame"
        )
    }


def open_video_at(
    video_path: Path,
    zero_based_index: int,
):
    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        int(
            zero_based_index
        ),
    )

    return cap


def read_exact_segment_frames(
    segment: pd.Series,
):
    cap = open_video_at(
        Path(
            segment[
                "video_path"
            ]
        ),
        int(
            segment[
                "video_start_index"
            ]
        ),
    )

    try:
        for local_idx in range(
            int(
                segment[
                    "length"
                ]
            )
        ):
            ok, frame = (
                cap.read()
            )

            if (
                not ok
                or frame is None
            ):
                raise RuntimeError(
                    "Video ended early at "
                    f"local frame {local_idx + 1}: "
                    f"{segment['video_path']}"
                )

            local_frame = (
                local_idx
                + 1
            )

            original_frame = (
                int(
                    segment[
                        "start_frame"
                    ]
                )
                + local_idx
            )

            yield (
                local_frame,
                original_frame,
                frame,
            )

    finally:
        cap.release()


def write_mot_gt(
    seq_name: str,
    segment: pd.Series,
    gt_df: pd.DataFrame,
    gt_benchmark_root: Path,
):
    seq_dir = (
        gt_benchmark_root
        / seq_name
    )

    gt_dir = (
        seq_dir
        / "gt"
    )

    gt_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    start_frame = int(
        segment[
            "start_frame"
        ]
    )

    lines = []

    # MOTChallenge requires positive target IDs.
    # Remapping VisDrone IDs to contiguous 1-based IDs
    # preserves the tracking identity task.
    original_ids = sorted(
        int(x)
        for x in (
            gt_df[
                "track_id"
            ]
            .unique()
            .tolist()
        )
    )

    gt_id_map = {
        original_id: index + 1
        for (
            index,
            original_id,
        ) in enumerate(
            original_ids
        )
    }

    write_json(
        seq_dir
        / "visdrone_to_mot_gt_id_map.json",
        gt_id_map,
    )

    for row in gt_df.itertuples():
        local_frame = (
            int(
                row.frame
            )
            - start_frame
            + 1
        )

        x = float(
            row.xmin
        )

        y = float(
            row.ymin
        )

        w = float(
            row.xmax
            - row.xmin
        )

        h = float(
            row.ymax
            - row.ymin
        )

        occlusion = int(
            row.occluded
        )

        if occlusion == 0:
            visibility = 1.0

        elif occlusion == 1:
            visibility = 0.5

        else:
            visibility = 0.2

        mot_id = gt_id_map[
            int(
                row.track_id
            )
        ]

        lines.append(
            f"{local_frame},"
            f"{mot_id},"
            f"{x:.3f},"
            f"{y:.3f},"
            f"{w:.3f},"
            f"{h:.3f},"
            f"1,1,"
            f"{visibility:.3f},"
            f"-1"
        )

    (
        gt_dir
        / "gt.txt"
    ).write_text(
        "\n".join(
            lines
        )
        + (
            "\n"
            if lines
            else ""
        ),
        encoding="utf-8",
    )

    seqinfo = (
        "[Sequence]\n"
        f"name={seq_name}\n"
        "imDir=img1\n"
        f"frameRate={float(segment['fps']):.6f}\n"
        f"seqLength={int(segment['length'])}\n"
        f"imWidth={int(segment['width'])}\n"
        f"imHeight={int(segment['height'])}\n"
        "imExt=.jpg\n"
    )

    (
        seq_dir
        / "seqinfo.ini"
    ).write_text(
        seqinfo,
        encoding="utf-8",
    )


def append_mot_prediction_line(
    lines: List[str],
    local_frame: int,
    track_id: int,
    box: Sequence[float],
    score: float,
):
    x1, y1, x2, y2 = map(
        float,
        box,
    )

    width = (
        x2 - x1
    )

    height = (
        y2 - y1
    )

    lines.append(
        f"{local_frame},"
        f"{track_id},"
        f"{x1:.3f},"
        f"{y1:.3f},"
        f"{width:.3f},"
        f"{height:.3f},"
        f"{score:.6f},"
        f"-1,-1,-1"
    )


def id_color(
    track_id: int,
):
    rng = (
        np.random.default_rng(
            int(
                track_id
            )
            + 12345
        )
    )

    return tuple(
        int(x)
        for x in rng.integers(
            60,
            240,
            size=3,
        ).tolist()
    )


def draw_tracking_frame(
    frame,
    tracks,
    gt_frame: Optional[pd.DataFrame],
    trails,
    model_name,
    tracker_name,
    local_frame,
    detector_ms,
    tracker_ms,
):
    """
    Draw tracking output.

    `tracks` intentionally has no TrackBatch type annotation
    so this utility cell is independent of notebook execution order.
    """

    canvas = (
        frame.copy()
    )

    # ---------------------------------------------------------
    # Ground truth
    # ---------------------------------------------------------
    if (
        SHOW_GROUND_TRUTH_ON_VIDEO
        and gt_frame is not None
    ):
        for row in (
            gt_frame.itertuples()
        ):
            x1, y1, x2, y2 = map(
                int,
                [
                    row.xmin,
                    row.ymin,
                    row.xmax,
                    row.ymax,
                ],
            )

            cv2.rectangle(
                canvas,
                (
                    x1,
                    y1,
                ),
                (
                    x2,
                    y2,
                ),
                (
                    235,
                    235,
                    235,
                ),
                1,
            )

            cv2.putText(
                canvas,
                f"GT {int(row.track_id)}",
                (
                    x1,
                    max(
                        15,
                        y1 - 4,
                    ),
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (
                    235,
                    235,
                    235,
                ),
                1,
                cv2.LINE_AA,
            )

    # ---------------------------------------------------------
    # Predictions
    # ---------------------------------------------------------
    for (
        box,
        track_id,
        score,
    ) in zip(
        tracks.boxes,
        tracks.ids,
        tracks.scores,
    ):
        x1, y1, x2, y2 = map(
            int,
            box,
        )

        color = id_color(
            int(
                track_id
            )
        )

        cv2.rectangle(
            canvas,
            (
                x1,
                y1,
            ),
            (
                x2,
                y2,
            ),
            color,
            2,
        )

        cv2.putText(
            canvas,
            (
                f"ID {int(track_id)} "
                f"{float(score):.2f}"
            ),
            (
                x1,
                max(
                    18,
                    y1 - 6,
                ),
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2,
            cv2.LINE_AA,
        )

        if SHOW_TRACK_TRAILS:
            center = (
                int(
                    (
                        x1 + x2
                    )
                    / 2
                ),
                int(
                    (
                        y1 + y2
                    )
                    / 2
                ),
            )

            trails[
                int(
                    track_id
                )
            ].append(
                center
            )

            points = list(
                trails[
                    int(
                        track_id
                    )
                ]
            )

            for (
                point_1,
                point_2,
            ) in zip(
                points[:-1],
                points[1:],
            ):
                cv2.line(
                    canvas,
                    point_1,
                    point_2,
                    color,
                    2,
                    cv2.LINE_AA,
                )

    # ---------------------------------------------------------
    # Diagnostic overlay
    # ---------------------------------------------------------
    core_ms = (
        detector_ms
        + tracker_ms
    )

    core_fps = (
        1000.0
        / core_ms
        if core_ms > 0
        else 0.0
    )

    info = [
        f"{model_name} + {tracker_name}",
        f"Frame: {local_frame}",
        f"Tracks: {len(tracks)}",
        f"Detector: {detector_ms:.2f} ms",
        f"Tracker: {tracker_ms:.2f} ms",
        (
            f"Core: {core_ms:.2f} ms "
            f"({core_fps:.1f} FPS)"
        ),
    ]

    y = 28

    for line in info:
        cv2.putText(
            canvas,
            line,
            (
                15,
                y,
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.62,
            (
                255,
                255,
                255,
            ),
            2,
            cv2.LINE_AA,
        )

        y += 24

    return canvas


print(
    "Metrics/MOT/video utilities: READY"
)

Metrics/MOT/video utilities: READY


## 13. Prepare GT and execute the 3-detector × 2-tracker experiment matrix
Each detector runs once per clip. Its saved detections are replayed into both trackers,
guaranteeing that ByteTrack and BoT-SORT receive identical detector outputs.

In [ ]:
TRACKING_BENCHMARK_NAME = "VISDRONE_MOT_STEP5"
TRACKING_SPLIT = "test_dev"
TRACKEVAL_GT_ROOT = TRACKEVAL_DIR / "mot_gt"
TRACKEVAL_TRACKERS_ROOT = TRACKEVAL_DIR / "mot_trackers"
TRACKEVAL_OUTPUT_ROOT = TRACKEVAL_DIR / "results"
GT_BENCHMARK_ROOT = TRACKEVAL_GT_ROOT / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}"
GT_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
(TRACKEVAL_GT_ROOT / "seqmaps").mkdir(parents=True, exist_ok=True)
TRACKEVAL_TRACKERS_ROOT.mkdir(parents=True, exist_ok=True); TRACKEVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

segment_gt_cache = {}
for _, segment in SELECTED_SEGMENTS.iterrows():
    seq = str(segment["sequence_name"]); gt_df = load_segment_gt(segment); segment_gt_cache[seq] = gt_df
    write_mot_gt(seq, segment, gt_df, GT_BENCHMARK_ROOT)
seqmap_path = TRACKEVAL_GT_ROOT / "seqmaps" / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}.txt"
seqmap_path.write_text("name\n" + "\n".join(SELECTED_SEGMENTS["sequence_name"].astype(str)) + "\n", encoding="utf-8")


def detection_cache_paths(model_name: str, seq: str):
    root = RUNS_DIR / safe_slug(model_name) / "detector_cache" / seq; root.mkdir(parents=True, exist_ok=True)
    return {"root": root, "jsonl": root/"detections.jsonl", "summary": root/"detection_summary.json",
            "timings": root/"detector_timings.csv", "complete": root/"_COMPLETE.json"}


def serialize_detection(local_frame, original_frame, b: DetectionBatch):
    return {"local_frame": int(local_frame), "original_frame": int(original_frame),
            "boxes": b.boxes.astype(float).tolist(), "scores": b.scores.astype(float).tolist(),
            "preprocess_ms": float(b.preprocess_ms), "inference_ms": float(b.inference_ms),
            "postprocess_ms": float(b.postprocess_ms), "total_ms": float(b.total_ms)}


def deserialize_detection(row):
    boxes = np.asarray(row.get("boxes", []), np.float32).reshape(-1,4); scores = np.asarray(row.get("scores", []), np.float32)
    return DetectionBatch(boxes, scores, np.zeros((len(boxes),), np.float32), float(row.get("preprocess_ms",0)),
                          float(row.get("inference_ms",0)), float(row.get("postprocess_ms",0)), float(row.get("total_ms",0)))


def load_detection_cache(path: Path):
    rows = [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    return rows, {int(r["original_frame"]): deserialize_detection(r) for r in rows}


def run_detector_for_segment(detector, model_name: str, segment: pd.Series):
    seq = str(segment["sequence_name"]); paths = detection_cache_paths(model_name, seq)
    if RESUME_COMPLETED_DETECTION_CACHE and paths["complete"].exists() and paths["jsonl"].exists() and paths["summary"].exists():
        print("Detection cache complete:", model_name, seq)
        rows, by_original = load_detection_cache(paths["jsonl"])
        return rows, by_original, read_json(paths["summary"])

    cap = open_video_at(Path(segment["video_path"]), int(segment["video_start_index"])); ok, first_frame = cap.read(); cap.release()
    if not ok or first_frame is None: raise RuntimeError(f"Could not read warmup frame: {seq}")
    for _ in range(WARMUP_RUNS): _ = detector.predict(first_frame)
    reset_cuda_peak(); rows, by_original, timing_rows = [], {}, []
    for local_frame, original_frame, frame in tqdm(read_exact_segment_frames(segment), total=int(segment["length"]),
                                                   desc=f"Detect {model_name} | {seq}", leave=False):
        b = detector.predict(frame); by_original[original_frame] = b
        row = serialize_detection(local_frame, original_frame, b); rows.append(row)
        timing_rows.append({"local_frame": local_frame, "original_frame": original_frame, "num_detections": len(b),
                            "preprocess_ms": b.preprocess_ms, "inference_ms": b.inference_ms,
                            "postprocess_ms": b.postprocess_ms, "total_ms": b.total_ms})
    det_metrics = detection_operating_metrics(segment_gt_cache[seq], by_original)
    total_times = [r["total_ms"] for r in timing_rows]; inf_times = [r["inference_ms"] for r in timing_rows]
    mean_total = nanmean(total_times)
    summary = {"model": model_name, "sequence_name": seq, "frames": len(rows),
               "mean_detector_total_ms": mean_total, "p50_detector_total_ms": percentile(total_times,50),
               "p95_detector_total_ms": percentile(total_times,95), "mean_detector_inference_ms": nanmean(inf_times),
               "detector_fps": 1000/mean_total if mean_total > 0 else float("nan"), "peak_cuda_mib": peak_cuda_mib(),
               **det_metrics}
    if SAVE_DETECTION_CACHE_JSONL:
        with paths["jsonl"].open("w", encoding="utf-8") as f:
            for row in rows: f.write(json.dumps(row) + "\n")
    pd.DataFrame(timing_rows).to_csv(paths["timings"], index=False); write_json(paths["summary"], summary)
    write_json(paths["complete"], {"timestamp_utc": utc_now_iso(), "frames": len(rows)})
    return rows, by_original, summary


def tracker_run_paths(model_name, tracker_name, seq):
    system = safe_slug(f"{model_name}__{tracker_name}")
    root = RUNS_DIR / system / seq
    root.mkdir(parents=True, exist_ok=True)
    return system, {
        "root": root,
        "video": root / "annotated_tracking.mp4",
        "frames_csv": root / "frame_metrics.csv",
        "tracks_jsonl": root / "tracks.jsonl",
        "summary": root / "run_summary.json",
        "mot_prediction": root / "mot_predictions.txt",
        "complete": root / "_COMPLETE.json",
    }

def run_tracker_for_segment(model_name, tracker_name, segment, detections_by_original, detector_summary):
    seq = str(segment["sequence_name"])
    system, paths = tracker_run_paths(model_name, tracker_name, seq)
    tracker_eval_dir = TRACKEVAL_TRACKERS_ROOT / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}" / system / "data"
    tracker_eval_dir.mkdir(parents=True, exist_ok=True)
    tracker_eval_mot = tracker_eval_dir / f"{seq}.txt"

    if (RESUME_COMPLETED_TRACKER_RUNS and paths["complete"].exists()
            and paths["summary"].exists() and tracker_eval_mot.exists()):
        print("Tracker run complete:", system, seq)
        return read_json(paths["summary"])

    fps = float(segment["fps"])
    width, height = int(segment["width"]), int(segment["height"])
    tracker = CommonTracker(model_name, tracker_name, fps)
    tracker_config_audit = tracker.audit_dict()
    print(
        f"{model_name} + {tracker_name} | FPS={fps:.3f} | "
        f"buffer={tracker_config_audit['actual_tracker_buffer_frames']} frames "
        f"({tracker_config_audit['actual_tracker_buffer_seconds']:.2f} s) | "
        f"match={tracker_config_audit['match_thresh']:.2f} | "
        f"high={tracker_config_audit['track_high_thresh']:.2f} | "
        f"new={tracker_config_audit['new_track_thresh']:.2f}"
    )

    gt_by = frame_gt_dict(segment_gt_cache[seq])
    writer = None
    if SAVE_ANNOTATED_VIDEO:
        writer = cv2.VideoWriter(
            str(paths["video"]), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
        )
        if not writer.isOpened():
            raise RuntimeError(f"Could not create video: {paths['video']}")

    trails = defaultdict(lambda: deque(maxlen=TRACK_TRAIL_LENGTH))
    frame_rows, track_rows, mot_lines = [], [], []
    tracker_times, core_times = [], []
    wall_start = time.perf_counter()
    try:
        for local_frame, original_frame, frame in tqdm(
            read_exact_segment_frames(segment), total=int(segment["length"]),
            desc=f"Track {system} | {seq}", leave=False
        ):
            det = detections_by_original[original_frame]
            t0 = time.perf_counter()
            tracks = tracker.update(det, frame)
            cuda_sync()
            tracker_ms = (time.perf_counter() - t0) * 1000.0
            core_ms = float(det.total_ms) + tracker_ms
            tracker_times.append(tracker_ms); core_times.append(core_ms)

            for box, tid, score in zip(tracks.boxes, tracks.ids, tracks.scores):
                append_mot_prediction_line(mot_lines, local_frame, int(tid), box, float(score))

            frame_rows.append({
                "local_frame": int(local_frame), "original_frame": int(original_frame),
                "detections": len(det), "tracks": len(tracks),
                "detector_ms": float(det.total_ms), "tracker_ms": float(tracker_ms),
                "core_pipeline_ms": float(core_ms),
                "core_pipeline_fps": 1000.0/core_ms if core_ms > 0 else np.nan,
            })
            track_rows.append({
                "local_frame": int(local_frame), "original_frame": int(original_frame),
                "track_ids": [int(x) for x in tracks.ids.tolist()],
                "boxes": tracks.boxes.astype(float).tolist(),
                "scores": tracks.scores.astype(float).tolist(),
            })
            if writer is not None:
                writer.write(draw_tracking_frame(
                    frame, tracks, gt_by.get(original_frame), trails,
                    model_name, tracker_name, local_frame, float(det.total_ms), tracker_ms
                ))
    finally:
        if writer is not None:
            writer.release()

    wall_s = time.perf_counter() - wall_start
    mot_text = "\n".join(mot_lines) + ("\n" if mot_lines else "")
    paths["mot_prediction"].write_text(mot_text, encoding="utf-8")
    tracker_eval_mot.write_text(mot_text, encoding="utf-8")

    if SAVE_FRAME_CSV:
        pd.DataFrame(frame_rows).to_csv(paths["frames_csv"], index=False)
    if SAVE_TRACK_JSONL:
        with paths["tracks_jsonl"].open("w", encoding="utf-8") as f:
            for row in track_rows:
                f.write(json.dumps(row) + "\n")

    mean_tracker = nanmean(tracker_times); mean_core = nanmean(core_times)
    summary = {
        "system_name": system, "model": model_name, "tracker": tracker_name,
        "sequence_name": seq, "frames": len(frame_rows),
        "mean_tracker_ms": mean_tracker,
        "p50_tracker_ms": percentile(tracker_times, 50),
        "p95_tracker_ms": percentile(tracker_times, 95),
        "mean_core_pipeline_ms": mean_core,
        "p50_core_pipeline_ms": percentile(core_times, 50),
        "p95_core_pipeline_ms": percentile(core_times, 95),
        "core_pipeline_fps": 1000.0/mean_core if mean_core > 0 else np.nan,
        "tracker_overhead_fraction": mean_tracker/mean_core if mean_core > 0 else np.nan,
        "annotated_export_wall_fps": len(frame_rows)/wall_s if wall_s > 0 else np.nan,
        "detector_peak_cuda_mib": float(detector_summary.get("peak_cuda_mib", np.nan)),
        "tracker_config": tracker_config_audit,
        "track_buffer_frames": int(tracker_config_audit["actual_tracker_buffer_frames"]),
        "track_buffer_seconds": float(tracker_config_audit["actual_tracker_buffer_seconds"]),
        "mot_prediction_path": str(tracker_eval_mot),
        "annotated_video_path": str(paths["video"]) if SAVE_ANNOTATED_VIDEO else None,
    }
    write_json(paths["summary"], summary)
    write_json(paths["complete"], {
        "timestamp_utc": utc_now_iso(), "frames": len(frame_rows), "reid_used": False
    })
    return summary

MODELS_TO_RUN = ["YOLO26s", "RT-DETR-R18", "BPD-YOLOn/L-FPN"]
SYSTEM_ERRORS, detector_summary_rows, tracker_summary_rows = {}, [], []
for model_name in MODELS_TO_RUN:
    print("\n" + "#"*100); print("BUILDING DETECTOR:", model_name); print("#"*100)
    detector = None
    try:
        detector = build_detector(model_name)
        for _, segment in SELECTED_SEGMENTS.iterrows():
            _, det_by_original, det_summary = run_detector_for_segment(detector, model_name, segment)
            detector_summary_rows.append(det_summary)
            for tracker_name in TRACKERS_TO_RUN:
                try:
                    tracker_summary_rows.append(run_tracker_for_segment(model_name, tracker_name, segment, det_by_original, det_summary))
                except Exception as exc:
                    key = f"{model_name}__{tracker_name}__{segment['sequence_name']}"; SYSTEM_ERRORS[key] = traceback.format_exc()
                    print("FAILED:", key, "\n", exc)
                    if not CONTINUE_ON_SYSTEM_ERROR: raise
    except Exception as exc:
        key = f"{model_name}__detector"; SYSTEM_ERRORS[key] = traceback.format_exc(); print("DETECTOR FAILED:", model_name, exc)
        if not CONTINUE_ON_SYSTEM_ERROR: raise
    finally:
        if detector is not None: del detector
        gc.collect(); torch.cuda.empty_cache()

DETECTOR_SUMMARIES = pd.DataFrame(detector_summary_rows); TRACKER_RUN_SUMMARIES = pd.DataFrame(tracker_summary_rows)
DETECTOR_SUMMARIES.to_csv(METRICS_DIR/"detector_segment_metrics.csv", index=False)
TRACKER_RUN_SUMMARIES.to_csv(METRICS_DIR/"tracker_runtime_segment_metrics.csv", index=False)
write_json(AUDIT_DIR/"system_errors.json", SYSTEM_ERRORS)
print("Detector/tracker stage finished. Errors:", len(SYSTEM_ERRORS))



####################################################################################################
BUILDING DETECTOR: YOLO26s
####################################################################################################
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading /content/step5_model_cache_v5/YOLO26s/4ab51e0cc1e09828/yolo26s_step5_model.engine for TensorRT inference...


Detect YOLO26s | visdrone_uav0000201_00000_v_f000315_to_000614:   0%|          | 0/300 [00:00<?, ?it/s]

YOLO26s + ByteTrack | FPS=30.000 | buffer=60 frames (2.00 s) | match=0.80 | high=0.30 | new=0.35


Track YOLO26s_ByteTrack | visdrone_uav0000201_00000_v_f000315_to_000614:   0%|          | 0/300 [00:00<?, ?it/…

YOLO26s + BoT-SORT | FPS=30.000 | buffer=60 frames (2.00 s) | match=0.80 | high=0.30 | new=0.35


...

BPD-YOLOn/L-FPN + ByteTrack | FPS=30.000 | buffer=60 frames (2.00 s) | match=0.80 | high=0.30 | new=0.35


Track BPD-YOLOn_L-FPN_ByteTrack | visdrone_uav0000161_00000_v_f000001_to_000300:   0%|          | 0/300 [00:00…

BPD-YOLOn/L-FPN + BoT-SORT | FPS=30.000 | buffer=60 frames (2.00 s) | match=0.80 | high=0.30 | new=0.35


Track BPD-YOLOn_L-FPN_BoT-SORT | visdrone_uav0000161_00000_v_f000001_to_000300:   0%|          | 0/300 [00:00<…

Detector/tracker stage finished. Errors: 0


## Official TrackEval — NumPy 2.x compatibility hotfix

TrackEval is now executed through its **direct Python API** rather than the
`run_mot_challenge.py` command-line wrapper.

This preserves the official TrackEval evaluator and metric implementations,
while fixing compatibility with modern NumPy and avoiding the CLI
`OUTPUT_FOLDER` list conversion issue.

The upstream TrackEval source repository is **not modified**.


In [ ]:
TRACKEVAL_REPO_ROOT = TOOL_CACHE / "TrackEval"

TRACK_EVAL_EXECUTED = False
TRACK_EVAL_SKIP_REASON = None
TRACK_EVAL_RESULTS = None


def prepare_trackeval_repository():
    """
    Clone TrackEval if needed and record the exact upstream commit.
    """
    if not TRACKEVAL_REPO_ROOT.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                TRACKEVAL_REPO_URL,
                str(TRACKEVAL_REPO_ROOT),
            ]
        )

    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(TRACKEVAL_REPO_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()

    write_json(
        AUDIT_DIR / "trackeval_source.json",
        {
            "repo": TRACKEVAL_REPO_URL,
            "resolved_commit": commit,
            "execution_mode": "direct_python_api",
            "numpy_compatibility_shim": True,
        },
    )

    return commit


def install_trackeval_numpy_compatibility_shim():
    """
    TrackEval upstream still contains deprecated NumPy aliases such as
    np.float and np.int.

    Modern NumPy removed those aliases.  They originally mapped to Python
    builtins, so this local-process compatibility shim restores only the
    missing names without downgrading NumPy or editing the TrackEval source.

    np.float64 / np.int64 and all normal NumPy scalar types are untouched.
    """
    compatibility_aliases = {
        "float": float,
        "int": int,
        "bool": bool,
        "object": object,
        "str": str,
        "complex": complex,
    }

    installed = {}

    for name, builtin_type in compatibility_aliases.items():
        if name not in np.__dict__:
            setattr(np, name, builtin_type)
            installed[name] = builtin_type.__name__

    write_json(
        AUDIT_DIR / "trackeval_numpy_compatibility.json",
        {
            "numpy_version": np.__version__,
            "installed_aliases": installed,
            "scope": (
                "Current notebook Python process only; "
                "upstream TrackEval source files are not modified."
            ),
        },
    )

    if installed:
        print(
            "TrackEval NumPy compatibility aliases installed:",
            installed,
        )
    else:
        print(
            "TrackEval NumPy compatibility shim: "
            "no missing aliases required."
        )

    return installed


def expected_system_names():
    return [
        safe_slug(f"{model_name}__{tracker_name}")
        for model_name in MODELS_TO_RUN
        for tracker_name in TRACKERS_TO_RUN
    ]


def trackeval_missing_predictions() -> List[str]:
    missing = []

    for system in expected_system_names():
        for seq in SELECTED_SEGMENTS["sequence_name"].astype(str):
            p = (
                TRACKEVAL_TRACKERS_ROOT
                / f"{TRACKING_BENCHMARK_NAME}-{TRACKING_SPLIT}"
                / system
                / "data"
                / f"{seq}.txt"
            )

            if not p.exists():
                missing.append(str(p))

    return missing


def import_trackeval_direct():
    """
    Import the cloned TrackEval repository into this notebook process.
    """
    repo_text = str(TRACKEVAL_REPO_ROOT)

    if repo_text not in sys.path:
        sys.path.insert(0, repo_text)

    importlib.invalidate_caches()

    import trackeval

    return trackeval


def run_trackeval():
    """
    Run TrackEval through its Python API instead of scripts/run_mot_challenge.py.

    Advantages:
      1. NumPy compatibility shim is active in the same process.
      2. OUTPUT_FOLDER remains a normal string instead of being converted
         to a one-element list by the CLI argparse path.
      3. The official TrackEval Evaluator, MotChallenge2DBox dataset class,
         HOTA, CLEAR and Identity implementations are still used.
    """
    global TRACK_EVAL_EXECUTED
    global TRACK_EVAL_SKIP_REASON
    global TRACK_EVAL_RESULTS

    missing = trackeval_missing_predictions()

    if SYSTEM_ERRORS or missing:
        TRACK_EVAL_SKIP_REASON = {
            "system_errors": SYSTEM_ERRORS,
            "missing_prediction_count": len(missing),
            "missing_prediction_sample": missing[:10],
        }

        write_json(
            TRACKEVAL_DIR / "TRACK_EVAL_SKIPPED.json",
            TRACK_EVAL_SKIP_REASON,
        )

        print("=" * 96)
        print("TRACKEVAL SKIPPED")
        print("=" * 96)
        print(
            "The detector/tracker matrix is incomplete, "
            "so TrackEval was not started."
        )
        print("System errors            :", len(SYSTEM_ERRORS))
        print("Missing prediction files :", len(missing))
        print("=" * 96)

        return False

    commit = prepare_trackeval_repository()

    install_trackeval_numpy_compatibility_shim()

    trackeval = import_trackeval_direct()

    systems = expected_system_names()

    # --------------------------------------------------------
    # Evaluator configuration
    # --------------------------------------------------------

    eval_config = trackeval.Evaluator.get_default_eval_config()

    eval_config.update(
        {
            "USE_PARALLEL": False,
            "NUM_PARALLEL_CORES": 1,
            "BREAK_ON_ERROR": True,
            "RETURN_ON_ERROR": False,
            "PRINT_RESULTS": True,
            "PRINT_ONLY_COMBINED": False,
            "PRINT_CONFIG": True,
            "TIME_PROGRESS": True,
            "DISPLAY_LESS_PROGRESS": False,
            "OUTPUT_SUMMARY": True,
            "OUTPUT_EMPTY_CLASSES": True,
            "OUTPUT_DETAILED": True,
            "PLOT_CURVES": True,
        }
    )

    # --------------------------------------------------------
    # MOTChallenge-compatible custom Okutama configuration
    # --------------------------------------------------------

    dataset_config = (
        trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    )

    dataset_config.update(
        {
            "GT_FOLDER": str(TRACKEVAL_GT_ROOT),
            "TRACKERS_FOLDER": str(TRACKEVAL_TRACKERS_ROOT),

            # IMPORTANT:
            # Direct Python API keeps this as a string.
            "OUTPUT_FOLDER": str(TRACKEVAL_OUTPUT_ROOT),

            "TRACKERS_TO_EVAL": systems,
            "CLASSES_TO_EVAL": ["pedestrian"],
            "BENCHMARK": TRACKING_BENCHMARK_NAME,
            "SPLIT_TO_EVAL": TRACKING_SPLIT,
            "INPUT_AS_ZIP": False,
            "PRINT_CONFIG": True,
            "DO_PREPROC": False,
            "TRACKER_SUB_FOLDER": "data",
            "OUTPUT_SUB_FOLDER": "",

            # Use the exact seqmap generated by this notebook.
            "SEQMAP_FOLDER": str(
                TRACKEVAL_GT_ROOT / "seqmaps"
            ),
            "SEQMAP_FILE": str(seqmap_path),

            "SEQ_INFO": None,
            "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
            "SKIP_SPLIT_FOL": False,
        }
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics_config = {
        "METRICS": [
            "HOTA",
            "CLEAR",
            "Identity",
        ],
        "THRESHOLD": 0.5,
    }

    evaluator = trackeval.Evaluator(
        eval_config
    )

    dataset_list = [
        trackeval.datasets.MotChallenge2DBox(
            dataset_config
        )
    ]

    metrics_list = [
        trackeval.metrics.HOTA(
            metrics_config
        ),
        trackeval.metrics.CLEAR(
            metrics_config
        ),
        trackeval.metrics.Identity(
            metrics_config
        ),
    ]

    print("=" * 96)
    print("RUNNING TRACKEVAL THROUGH DIRECT PYTHON API")
    print("=" * 96)
    print("TrackEval commit :", commit)
    print("NumPy version    :", np.__version__)
    print("Systems          :", systems)
    print("Sequences        :", SELECTED_SEGMENTS["sequence_name"].tolist())
    print("Output folder    :", TRACKEVAL_OUTPUT_ROOT)
    print("=" * 96)

    start_time = time.perf_counter()

    try:
        TRACK_EVAL_RESULTS = evaluator.evaluate(
            dataset_list,
            metrics_list,
        )

    except Exception:
        error_text = traceback.format_exc()

        (
            TRACKEVAL_DIR
            / "trackeval_direct_api_error.log"
        ).write_text(
            error_text,
            encoding="utf-8",
        )

        raise RuntimeError(
            "TrackEval direct Python API failed. "
            f"See {TRACKEVAL_DIR / 'trackeval_direct_api_error.log'}"
        )

    elapsed_s = (
        time.perf_counter()
        - start_time
    )

    write_json(
        TRACKEVAL_DIR / "trackeval_execution.json",
        {
            "status": "PASSED",
            "execution_mode": "direct_python_api",
            "commit": commit,
            "numpy_version": np.__version__,
            "elapsed_seconds": elapsed_s,
            "systems": systems,
            "sequences": (
                SELECTED_SEGMENTS[
                    "sequence_name"
                ]
                .astype(str)
                .tolist()
            ),
        },
    )

    TRACK_EVAL_EXECUTED = True

    print("=" * 96)
    print("TRACKEVAL: PASSED")
    print(f"Elapsed: {elapsed_s:.2f} s")
    print("=" * 96)

    return True


run_trackeval()

TrackEval NumPy compatibility aliases installed: {'float': 'float', 'int': 'int', 'object': 'object', 'str': 'str', 'complex': 'complex'}

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/step5_tools/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
GT_FOLDER            : /content/drive/MyDrive/aeri

True

<Figure size 640x480 with 0 Axes>

## 15. Build the final Step-5 metric matrix

In [ ]:
if not globals().get("TRACK_EVAL_EXECUTED", False):
    raise RuntimeError(
        "Final Step-5 metric matrix was not built because TrackEval did not run. "
        "Fix the upstream detector/tracker failure first; see "
        f"{TRACKEVAL_DIR / 'TRACK_EVAL_SKIPPED.json'} if it exists."
    )

def read_trackeval_summary_file(path: Path) -> Dict[str, float]:
    lines = [x.strip() for x in path.read_text(encoding="utf-8", errors="replace").splitlines() if x.strip()]
    if len(lines) < 2: raise RuntimeError(f"Unexpected TrackEval summary: {path}")
    headers, values = lines[0].split(), lines[1].split()
    if len(headers) != len(values): raise RuntimeError(f"Header/value mismatch: {path}")
    out = {}
    for k, v in zip(headers, values):
        try: out[k] = float(v)
        except ValueError: pass
    return out


PERCENT_LIKE_KEYS = {"HOTA","DetA","AssA","LocA","MOTA","MOTP","MODA","CLR_Re","CLR_Pr","MTR","PTR","MLR","sMOTA","IDF1","IDR","IDP"}
def normalize_trackeval_fraction(k, v):
    return v/100.0 if k in PERCENT_LIKE_KEYS and np.isfinite(v) and abs(v) > 1.5 else v


summary_files = sorted(TRACKEVAL_OUTPUT_ROOT.rglob("*_summary.txt"))
if not summary_files:
    summary_files = sorted(TRACKEVAL_TRACKERS_ROOT.rglob("*_summary.txt"))
if not summary_files:
    raise RuntimeError("TrackEval completed but no *_summary.txt files were found")

tracking_rows = []
for path in summary_files:
    system = next((x for x in expected_system_names() if x in str(path)), None)
    if system is None: continue
    metrics = {k: normalize_trackeval_fraction(k, v) for k, v in read_trackeval_summary_file(path).items()}
    tracking_rows.append({"system_name": system, "summary_file": str(path), **metrics})
TRACKING_METRICS = pd.DataFrame(tracking_rows).sort_values("system_name").drop_duplicates("system_name", keep="last").reset_index(drop=True)
if TRACKING_METRICS.empty: raise RuntimeError("Could not map TrackEval summaries to Step-5 systems")

lookup = {safe_slug(f"{m}__{t}"): (m,t) for m in MODELS_TO_RUN for t in TRACKERS_TO_RUN}
TRACKING_METRICS["model"] = TRACKING_METRICS["system_name"].map(lambda x: lookup[x][0])
TRACKING_METRICS["tracker"] = TRACKING_METRICS["system_name"].map(lambda x: lookup[x][1])
if "IDSW" not in TRACKING_METRICS.columns:
    for alt in ["IDs", "IDSw"]:
        if alt in TRACKING_METRICS.columns: TRACKING_METRICS["IDSW"] = TRACKING_METRICS[alt]; break
if "Frag" not in TRACKING_METRICS.columns:
    for alt in ["FRAG", "Fragments"]:
        if alt in TRACKING_METRICS.columns: TRACKING_METRICS["Frag"] = TRACKING_METRICS[alt]; break
if "IDSW" not in TRACKING_METRICS.columns: TRACKING_METRICS["IDSW"] = np.nan
if "Frag" not in TRACKING_METRICS.columns: TRACKING_METRICS["Frag"] = np.nan

RUNTIME_AGG = TRACKER_RUN_SUMMARIES.groupby(
    ["system_name", "model", "tracker"], as_index=False
).agg(
    frames=("frames","sum"),
    mean_tracker_ms=("mean_tracker_ms","mean"),
    p50_tracker_ms=("p50_tracker_ms","mean"),
    p95_tracker_ms=("p95_tracker_ms","mean"),
    mean_core_pipeline_ms=("mean_core_pipeline_ms","mean"),
    p50_core_pipeline_ms=("p50_core_pipeline_ms","mean"),
    p95_core_pipeline_ms=("p95_core_pipeline_ms","mean"),
    core_pipeline_fps=("core_pipeline_fps","mean"),
    tracker_overhead_fraction=("tracker_overhead_fraction","mean"),
    annotated_export_wall_fps=("annotated_export_wall_fps","mean"),
    detector_peak_cuda_mib=("detector_peak_cuda_mib","max"),
)
FINAL_TRACKING_MATRIX = TRACKING_METRICS.merge(RUNTIME_AGG, on=["system_name","model","tracker"], how="left")
FINAL_TRACKING_MATRIX["idsw_per_100_frames"] = FINAL_TRACKING_MATRIX["IDSW"] / FINAL_TRACKING_MATRIX["frames"].clip(lower=1) * 100
FINAL_TRACKING_MATRIX["fragments_per_100_frames"] = FINAL_TRACKING_MATRIX["Frag"] / FINAL_TRACKING_MATRIX["frames"].clip(lower=1) * 100
FINAL_TRACKING_MATRIX.to_csv(METRICS_DIR/"tracking_metrics_all_systems.csv", index=False)
write_json(METRICS_DIR/"tracking_metrics_all_systems.json", FINAL_TRACKING_MATRIX.to_dict("records"))


DETECTOR_AGG = DETECTOR_SUMMARIES.groupby("model", as_index=False).agg(
    frames=("frames","sum"), detector_fps=("detector_fps","mean"), mean_detector_total_ms=("mean_detector_total_ms","mean"),
    p95_detector_total_ms=("p95_detector_total_ms","mean"), peak_cuda_mib=("peak_cuda_mib","max"),
    precision=("precision","mean"), recall=("recall","mean"), f1=("f1","mean"),
    visible_recall=("visible_recall","mean"), occluded_recall=("occluded_recall","mean"),
    h_lt16_recall=("h_lt16_recall","mean"), h_16_31_recall=("h_16_31_recall","mean"),
    h_32_95_recall=("h_32_95_recall","mean"), h_ge96_recall=("h_ge96_recall","mean")
)
DETECTOR_AGG.to_csv(METRICS_DIR/"detector_metrics_aggregate.csv", index=False)

# Raw per-sequence detailed TrackEval tables when available.
detailed = []
for path in TRACKEVAL_OUTPUT_ROOT.rglob("*_detailed.csv"):
    system = next((x for x in expected_system_names() if x in str(path)), None)
    if system is None: continue
    try: df = pd.read_csv(path)
    except Exception: continue
    df["system_name"] = system; df["source_file"] = str(path); detailed.append(df)
if detailed:
    pd.concat(detailed, ignore_index=True, sort=False).to_csv(METRICS_DIR/"tracking_metrics_per_sequence_raw.csv", index=False)

# BoT-SORT minus ByteTrack delta per detector.
delta_rows = []
for model_name in MODELS_TO_RUN:
    d = FINAL_TRACKING_MATRIX[FINAL_TRACKING_MATRIX["model"] == model_name].set_index("tracker")
    if not {"ByteTrack","BoT-SORT"}.issubset(d.index): continue
    bt, bs = d.loc["ByteTrack"], d.loc["BoT-SORT"]; rec = {"model": model_name}
    for metric in ["HOTA","DetA","AssA","MOTA","MOTP","IDF1","IDSW","Frag","idsw_per_100_frames"]:
        if metric in d.columns: rec[f"delta_botsort_minus_bytetrack__{metric}"] = float(bs[metric])-float(bt[metric])
    delta_rows.append(rec)
TRACKER_DELTA = pd.DataFrame(delta_rows); TRACKER_DELTA.to_csv(METRICS_DIR/"tracker_delta_botsort_minus_bytetrack.csv", index=False)

show_cols = [
    c
    for c in [
        "model",
        "tracker",
        "HOTA",
        "DetA",
        "AssA",
        "MOTA",
        "MOTP",
        "IDF1",
        "IDSW",
        "Frag",
        "idsw_per_100_frames",
    ]
    if c in FINAL_TRACKING_MATRIX.columns
]
display(FINAL_TRACKING_MATRIX[show_cols].sort_values(["model","tracker"]).reset_index(drop=True))


,model,tracker,HOTA,DetA,AssA,MOTA,MOTP,IDF1,IDSW,Frag,idsw_per_100_frames
0,BPD-YOLOn/L-FPN,BoT-SORT,0.27378,0.18310,0.41201,-1.22790,0.71800,0.30716,237.0,528.0,15.800000
1,BPD-YOLOn/L-FPN,ByteTrack,0.25112,0.18767,0.33894,-1.09580,0.71789,0.27499,282.0,507.0,18.800000
2,RT-DETR-R18,BoT-SORT,0.28294,0.17724,0.45860,-1.44550,0.69594,0.31328,268.0,590.0,17.866667
3,RT-DETR-R18,ByteTrack,0.25292,0.18302,0.35840,-1.31450,0.69491,0.27057,378.0,587.0,25.200000
4,YOLO26s,BoT-SORT,0.29671,0.19871,0.44573,-1.00230,0.72658,0.33393,186.0,581.0,12.400000
5,YOLO26s,ByteTrack,0.26325,0.20348,0.34427,-0.88839,0.72638,0.29291,275.0,567.0,18.333333


## 16. Proposal-reference acceptance checks and report-ready plots
These checks report the project gates; they are not used to tune the Okutama test clips.

In [19]:
acceptance_rows = []

for _, row in FINAL_TRACKING_MATRIX.iterrows():
    mota = float(
        row.get(
            "MOTA",
            np.nan,
        )
    )

    idf1 = float(
        row.get(
            "IDF1",
            np.nan,
        )
    )

    idsw100 = float(
        row.get(
            "idsw_per_100_frames",
            np.nan,
        )
    )

    rec = {
        "system_name": row["system_name"],
        "model": row["model"],
        "tracker": row["tracker"],

        "MOTA": mota,
        "target_MOTA": TARGET_MOTA,
        "pass_MOTA": bool(
            np.isfinite(mota)
            and mota >= TARGET_MOTA
        ),

        "IDF1": idf1,
        "target_IDF1": TARGET_IDF1,
        "pass_IDF1": bool(
            np.isfinite(idf1)
            and idf1 >= TARGET_IDF1
        ),

        "IDSW_per_100_frames": idsw100,
        "target_max_IDSW_per_100_frames": (
            TARGET_IDSW_PER_100_FRAMES
        ),
        "pass_IDSW": bool(
            np.isfinite(idsw100)
            and idsw100
            <= TARGET_IDSW_PER_100_FRAMES
        ),
    }

    rec[
        "pass_all_reference_checks"
    ] = all(
        [
            rec["pass_MOTA"],
            rec["pass_IDF1"],
            rec["pass_IDSW"],
        ]
    )

    acceptance_rows.append(
        rec
    )


ACCEPTANCE = pd.DataFrame(
    acceptance_rows
)

ACCEPTANCE.to_csv(
    METRICS_DIR
    / "proposal_reference_acceptance_checks.csv",
    index=False,
)

def system_labels(df): return [f"{m}\n{t}" for m,t in zip(df["model"], df["tracker"])]
def save_bar(metric, ylabel, filename):
    if metric not in FINAL_TRACKING_MATRIX.columns: return
    df = FINAL_TRACKING_MATRIX.sort_values(["model","tracker"]).reset_index(drop=True)
    values = pd.to_numeric(df[metric], errors="coerce").to_numpy()
    fig, ax = plt.subplots(figsize=(12,6)); ax.bar(system_labels(df), values); ax.set_ylabel(ylabel)
    ax.set_title(f"Step-5 Tracking Comparison — {metric}"); ax.tick_params(axis="x", rotation=25); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/filename, dpi=180); plt.close(fig)


save_bar("HOTA","HOTA","01_hota_comparison.png"); save_bar("IDF1","IDF1","02_idf1_comparison.png")
save_bar("MOTA","MOTA","03_mota_comparison.png"); save_bar("idsw_per_100_frames","ID switches / 100 frames","04_idsw_per_100_frames.png")




if not DETECTOR_AGG.empty:
    plot_df = DETECTOR_AGG.set_index("model")[["recall","visible_recall","occluded_recall"]]
    fig, ax = plt.subplots(figsize=(10,6)); plot_df.plot(kind="bar", ax=ax); ax.set_ylabel("Recall at IoU operating point")
    ax.set_title("Detector Recall on Selected VisDrone2019-MOT Clips"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/"09_detector_visible_occluded_recall.png", dpi=180); plt.close(fig)
    size_df = DETECTOR_AGG.set_index("model")[["h_lt16_recall","h_16_31_recall","h_32_95_recall","h_ge96_recall"]]
    fig, ax = plt.subplots(figsize=(11,6)); size_df.plot(kind="bar", ax=ax); ax.set_ylabel("Recall at IoU operating point")
    ax.set_title("Detector Recall by GT Person Height"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.25)
    fig.tight_layout(); fig.savefig(PLOTS_DIR/"10_detector_size_aware_recall.png", dpi=180); plt.close(fig)

display(ACCEPTANCE)


,system_name,model,tracker,MOTA,target_MOTA,pass_MOTA,IDF1,target_IDF1,pass_IDF1,IDSW_per_100_frames,target_max_IDSW_per_100_frames,pass_IDSW,pass_all_reference_checks
0,BPD-YOLOn_L-FPN_BoT-SORT,BPD-YOLOn/L-FPN,BoT-SORT,-1.22790,0.5,False,0.30716,0.6,False,15.800000,5.0,False,False
1,BPD-YOLOn_L-FPN_ByteTrack,BPD-YOLOn/L-FPN,ByteTrack,-1.09580,0.5,False,0.27499,0.6,False,18.800000,5.0,False,False
2,RT-DETR-R18_BoT-SORT,RT-DETR-R18,BoT-SORT,-1.44550,0.5,False,0.31328,0.6,False,17.866667,5.0,False,False
3,RT-DETR-R18_ByteTrack,RT-DETR-R18,ByteTrack,-1.31450,0.5,False,0.27057,0.6,False,25.200000,5.0,False,False
4,YOLO26s_BoT-SORT,YOLO26s,BoT-SORT,-1.00230,0.5,False,0.33393,0.6,False,12.400000,5.0,False,False
5,YOLO26s_ByteTrack,YOLO26s,ByteTrack,-0.88839,0.5,False,0.29291,0.6,False,18.333333,5.0,False,False


## 17. Automatic Step-5 report package and SHA-256 manifest

In [ ]:
def best_row(metric, maximize=True):
    if metric not in FINAL_TRACKING_MATRIX.columns: return None
    vals = pd.to_numeric(FINAL_TRACKING_MATRIX[metric], errors="coerce")
    if not vals.notna().any(): return None
    return FINAL_TRACKING_MATRIX.loc[vals.idxmax() if maximize else vals.idxmin()]


leaders = {}
for metric, maximize in [("HOTA",True),("IDF1",True),("MOTA",True),("IDSW",False)]:
    r = best_row(metric, maximize)
    if r is not None: leaders[metric] = {"system_name": r["system_name"], "model": r["model"], "tracker": r["tracker"], "value": float(r[metric])}

expected_system_count = len(MODELS_TO_RUN)*len(TRACKERS_TO_RUN); actual_system_count = FINAL_TRACKING_MATRIX["system_name"].nunique()
expected_sequence_count = len(SELECTED_SEGMENTS); expected_run_count = expected_system_count*expected_sequence_count; actual_run_count = len(TRACKER_RUN_SUMMARIES)
completeness = {"expected_systems": expected_system_count, "evaluated_systems": int(actual_system_count),
                "expected_sequences": expected_sequence_count, "expected_tracker_segment_runs": expected_run_count,
                "completed_tracker_segment_runs": int(actual_run_count), "system_errors": len(SYSTEM_ERRORS),
                "complete": bool(actual_system_count==expected_system_count and actual_run_count==expected_run_count and not SYSTEM_ERRORS)}
write_json(AUDIT_DIR/"completion_audit.json", completeness)

FINAL_TRACKING_MATRIX.to_csv(REPORT_ASSETS_DIR/"table_tracking_main.csv", index=False)
DETECTOR_AGG.to_csv(REPORT_ASSETS_DIR/"table_detector_component.csv", index=False)
TRACKER_DELTA.to_csv(REPORT_ASSETS_DIR/"table_tracker_delta.csv", index=False)
ACCEPTANCE.to_csv(REPORT_ASSETS_DIR/"table_acceptance_checks.csv", index=False)
SELECTED_SEGMENTS.to_csv(REPORT_ASSETS_DIR/"table_selected_visdrone_mot_clips.csv", index=False)

report_cols = [
    c
    for c in [
        "model",
        "tracker",
        "HOTA",
        "DetA",
        "AssA",
        "MOTA",
        "MOTP",
        "IDF1",
        "IDSW",
        "Frag",
        "idsw_per_100_frames",
    ]
    if c in FINAL_TRACKING_MATRIX.columns
]
report_table = FINAL_TRACKING_MATRIX[report_cols].sort_values(["model","tracker"]).reset_index(drop=True)
lines = [
    "# STEP 5 — Multi-Object Tracking Integration Report", "", f"Generated: {utc_now_iso()}", "",
    "## Scope", "", "- Three fixed final aerial-person detectors.", "- ByteTrack and BoT-SORT without any auxiliary ReID neural model.",
    "- Identical cached detector outputs are replayed into both trackers.", "- Lost tracks are retained for 2 seconds; no appearance/ReID network is used.",
    "- VisDrone2019-MOT test-dev continuous aerial clips with official tracking IDs.",
    "- HOTA/CLEAR/Identity metrics are computed with TrackEval.", "- FPS/latency are retained only as internal diagnostics; Step 4 is the deployment-performance benchmark.", "- No private project Final Test is used.",
    "- Tracker settings are unchanged from the no-ReID Step-5 protocol; only the dataset is changed.", "", "## Environment", "",
    f"- GPU: {GPU_NAME}", f"- PyTorch: {torch.__version__}", f"- CUDA: {torch.version.cuda}",
    f"- Ultralytics: {ultralytics.__version__}", "", "## VisDrone2019-MOT protocol", "",
    f"- Dataset: VisDrone2019-MOT {VISDRONE_MOT_SPLIT}", f"- Selected videos: {SELECTED_SEGMENTS['video_key'].nunique()}",
    f"- Selected clips: {len(SELECTED_SEGMENTS)}", f"- Label mode(s): {', '.join(sorted(SELECTED_SEGMENTS['label_mode'].unique()))}",
    f"- Consistent tracking IDs available: {bool(SELECTED_SEGMENTS['consistent_ids'].all())}", "",
    "Official VisDrone2019-MOT pedestrian annotations (category 1) are used with their temporal target IDs.",
    "", "## Main tracking matrix", "", report_table.to_markdown(index=False), "", "## Metric leaders (descriptive only)", ""
]
for metric, info in leaders.items(): lines.append(f"- {metric}: {info['model']} + {info['tracker']} = {info['value']:.4f}")
lines += [
    "",
    "These leaders are not a final product winner; Step 6 performs the final multi-criteria comparison.",
    "",

          "## Completion audit", "", "```json", json.dumps(completeness, indent=2), "```", "",
          "## Important interpretation", "", "Step 5 measures detector-to-tracker integration and identity/association behavior on continuous aerial video. It does not replace the later independent final detection benchmark or the final RTX 3070 deployment benchmark.", ""]
(OUTPUT_ROOT/"STEP5_RUN_REPORT.md").write_text("\n".join(lines), encoding="utf-8")
write_json(OUTPUT_ROOT/"STEP5_RUN_REPORT.json", {
    "generated_utc": utc_now_iso(), "output_root": str(OUTPUT_ROOT), "environment": environment,
    "execution_config": execution_config, "dataset_summary": dataset_summary, "leaders": leaders,
    "completeness": completeness, "main_metrics": FINAL_TRACKING_MATRIX.to_dict("records"),
    "detector_metrics": DETECTOR_AGG.to_dict("records"), "tracker_delta": TRACKER_DELTA.to_dict("records"),
    "acceptance_checks": ACCEPTANCE.to_dict("records"),
})

readme = f'''# Step-5 Delivery Package

Persistent output root:
`{OUTPUT_ROOT}`

Most important report files:
1. STEP5_RUN_REPORT.md
2. 05_metrics/tracking_metrics_all_systems.csv
3. 05_metrics/detector_metrics_aggregate.csv
4. 05_metrics/tracker_delta_botsort_minus_bytetrack.csv
5. 05_metrics/proposal_reference_acceptance_checks.csv
6. 02_dataset/selected_segments.csv
7. 06_plots/*.png
8. 03_runs/*/*/annotated_tracking.mp4
9. 04_trackeval/trackeval_console.log
10. 00_audit/environment.json
11. 01_models/model_source_manifest.csv
12. MANIFEST_SHA256.csv

Do not report a final project winner from Step 5 alone.
'''
(OUTPUT_ROOT/"README_OUTPUTS.md").write_text(readme, encoding="utf-8")

manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if not path.is_file() or path.name == "MANIFEST_SHA256.csv": continue
    try: digest = sha256_file(path)
    except Exception as exc: digest = f"ERROR:{exc}"
    manifest_rows.append({"relative_path": str(path.relative_to(OUTPUT_ROOT)), "size_bytes": path.stat().st_size, "sha256": digest})
MANIFEST = pd.DataFrame(manifest_rows); MANIFEST.to_csv(OUTPUT_ROOT/"MANIFEST_SHA256.csv", index=False)

print("="*100); print("STEP 5 FINAL PACKAGE CREATED"); print("="*100)
print("Output root:", OUTPUT_ROOT); print("Systems:", actual_system_count, "/", expected_system_count)
print("Run matrix:", actual_run_count, "/", expected_run_count); print("Errors:", len(SYSTEM_ERRORS)); print("Complete:", completeness["complete"])
print("="*100)
if FAIL_IF_FINAL_MATRIX_INCOMPLETE and not completeness["complete"]:
    raise RuntimeError("Outputs were saved, but the Step-5 matrix is incomplete. Inspect 00_audit/system_errors.json and rerun.")


STEP 5 FINAL PACKAGE CREATED
Output root: /content/drive/MyDrive/aerial_human_detection/step5_tracking_final/visdrone_mot_l4_tuned_s100
Systems: 6 / 6
Run matrix: 30 / 30
Errors: 0
Complete: True


## Final configuration summary

Dataset: VisDrone2019-MOT test-dev. Person-only target: category 1 (pedestrian). Five seeded sequences per run. Only the three project detectors are used. ReID is disabled. TrackEval uses raw tracker IDs.